# Retail Sales Forecasting and Evaluation

## Purpose
Evaluate monthly retail sales forecasts using chronological holdout testing
and rolling-origin validation.

## Business questions
- How accurately do baseline forecasts predict monthly sales?
- Does a seasonal-naive forecast improve on a naive forecast?
- Does blending forecasts improve evaluation metrics?
- How do additive ETS and damped-trend ETS compare?

## Evaluation principles
- Keep training and validation periods chronological.
- Compare models on matching actual months.
- Interpret MAE, RMSE, WAPE, and bias together.
- Treat the fixed holdout and rolling-origin results as separate evaluations.

The input is the project's complete monthly sales dataset. This notebook is
a refactored copy of the forecasting analysis; the original notebook remains
unchanged.


In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import statsmodels
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Find the project root independently of the notebook's working directory.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate project root. Expected pyproject.toml and src/."
    )

INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "monthly_sales_complete.csv"
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Monthly sales input not found: {INPUT_PATH}")

monthly_ts = pd.read_csv(INPUT_PATH, parse_dates=["month"])
# Retain the input metadata columns required by the original QA and modeling steps.
monthly_ts = monthly_ts.sort_values("month").reset_index(drop=True)

# Basic input validation before forecasting.
assert monthly_ts["month"].notna().all(), "Missing month values."
assert monthly_ts["total_sales"].notna().all(), "Missing sales values."
assert not monthly_ts["month"].duplicated().any(), "Duplicate monthly dates."
assert monthly_ts["month"].is_monotonic_increasing, "Months are not chronological."
assert len(monthly_ts) == 55, f"Expected 55 complete months, got {len(monthly_ts)}."
assert monthly_ts["month"].min() == pd.Timestamp("2013-01-01")
assert monthly_ts["month"].max() == pd.Timestamp("2017-07-01")

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_PATH)
print("Monthly observations:", len(monthly_ts))
print("Period:", monthly_ts["month"].min().date(), "to", monthly_ts["month"].max().date())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("statsmodels:", statsmodels.__version__)


Project root: d:\GitHub\retail-sales-forecasting
Input: d:\GitHub\retail-sales-forecasting\data\processed\monthly_sales_complete.csv
Monthly observations: 55
Period: 2013-01-01 to 2017-07-01
pandas: 3.0.6
NumPy: 2.5.3
statsmodels: 0.15.0


## Phase 3.9 — Forecasting Readiness & Time-Series Validation Design
### Step 3.9.1 — Establish the forecasting specification

### Step 3.9.2 — Inspect the available forecasting history

In [3]:
forecasting_spec = {
    "target": "total_sales",
    "time_grain": "Monthly",
    "business_grain": "Total Retail Business",
    "first_complete_month": monthly_ts["month"].min(),
    "last_complete_month": monthly_ts["month"].max(),
    "available_observations": len(monthly_ts),
    "proposed_forecast_horizon": 3,
    "partial_months_excluded": True,
    "validation_method": "Chronological holdout",
    "initial_baselines": [
        "Naive",
        "Seasonal Naive"
    ],
    "evaluation_metrics": [
        "MAE",
        "RMSE",
        "WAPE",
        "Bias"
    ]
}

forecasting_spec

{'target': 'total_sales',
 'time_grain': 'Monthly',
 'business_grain': 'Total Retail Business',
 'first_complete_month': Timestamp('2013-01-01 00:00:00'),
 'last_complete_month': Timestamp('2017-07-01 00:00:00'),
 'available_observations': 55,
 'proposed_forecast_horizon': 3,
 'partial_months_excluded': True,
 'validation_method': 'Chronological holdout',
 'initial_baselines': ['Naive', 'Seasonal Naive'],
 'evaluation_metrics': ['MAE', 'RMSE', 'WAPE', 'Bias']}

In [4]:
monthly_ts[
    ["month", "total_sales"]
].tail(18)

,month,total_sales
37,2016-02-01,2.194741e+07
38,2016-03-01,2.313178e+07
39,2016-04-01,2.596303e+07
40,2016-05-01,2.477943e+07
41,2016-06-01,2.220922e+07
42,2016-07-01,2.346267e+07
43,2016-08-01,2.245241e+07
44,2016-09-01,2.241745e+07
45,2016-10-01,2.403039e+07
46,2016-11-01,2.464264e+07


### Step 3.9.3 — Create the training and validation datasets

1. Define the cutoff

In [5]:
validation_start = pd.Timestamp("2017-05-01")
validation_end = pd.Timestamp("2017-07-01")

forecast_horizon = 3

2. Split chronologically

In [6]:
train_data = monthly_ts[
    monthly_ts["month"] < validation_start
].copy()

validation_data = monthly_ts[
    (monthly_ts["month"] >= validation_start)
    & (monthly_ts["month"] <= validation_end)
].copy()

3. Inspect the split

In [7]:
split_summary = {
    "training_rows": len(train_data),
    "validation_rows": len(validation_data),

    "training_start": train_data["month"].min(),
    "training_end": train_data["month"].max(),

    "validation_start": validation_data["month"].min(),
    "validation_end": validation_data["month"].max(),

    "forecast_horizon": forecast_horizon
}

split_summary

{'training_rows': 52,
 'validation_rows': 3,
 'training_start': Timestamp('2013-01-01 00:00:00'),
 'training_end': Timestamp('2017-04-01 00:00:00'),
 'validation_start': Timestamp('2017-05-01 00:00:00'),
 'validation_end': Timestamp('2017-07-01 00:00:00'),
 'forecast_horizon': 3}

4. Validate that no future information enters training

In [8]:
split_qa = {
    "training_period_correct": (
        train_data["month"].max()
        < validation_data["month"].min()
    ),

    "training_rows_correct": len(train_data) == 52,

    "validation_rows_correct": len(validation_data) == 3,

    "validation_horizon_correct": (
        len(validation_data) == forecast_horizon
    ),

    "no_training_validation_overlap": (
        set(train_data["month"]).isdisjoint(
            set(validation_data["month"])
        )
    ),

    "chronological_order_correct": (
        train_data["month"].is_monotonic_increasing
        and validation_data["month"].is_monotonic_increasing
    )
}

split_qa

{'training_period_correct': True,
 'training_rows_correct': True,
 'validation_rows_correct': True,
 'validation_horizon_correct': True,
 'no_training_validation_overlap': True,
 'chronological_order_correct': True}

5. Display the validation observations

In [9]:
validation_data[
    ["month", "total_sales"]
]

,month,total_sales
52,2017-05-01,2.691185e+07
53,2017-06-01,2.568282e+07
54,2017-07-01,2.701148e+07


### Phase 3.10 — Baseline Forecasting

Step 3.10.1 — Understand the two baseline methods

Before coding, let's understand the business logic.

Model A — Naive Forecast

The naive method assumes that the next forecast period will have the same Sales as the latest observed period.

For our project:

Last training month: April 2017

April 2017 Sales: approximately 25.90M

Therefore, the naive forecast will use April's Sales for May, June, and July 2017.

Model B — Seasonal Naive Forecast

Retail Sales may follow recurring calendar patterns.

The seasonal naive method assumes that the next month's Sales will equal the Sales from the same month one year earlier.

This method is useful as a benchmark because our exploratory analysis identified recurring monthly patterns.

However, it may not capture business growth or changes in demand.

Step 3.10.2 — Create the Naive Forecast

In [10]:
naive_forecast = validation_data[
    ["month", "total_sales"]
].copy()

naive_forecast = naive_forecast.rename(
    columns={
        "total_sales": "actual_sales"
    }
)

last_training_sales = train_data[
    "total_sales"
].iloc[-1]

naive_forecast["naive_forecast"] = last_training_sales

naive_forecast

,month,actual_sales,naive_forecast
52,2017-05-01,2.691185e+07,2.589531e+07
53,2017-06-01,2.568282e+07,2.589531e+07
54,2017-07-01,2.701148e+07,2.589531e+07


### Step 3.10.3 — Create the Seasonal Naive Forecast

In [11]:
seasonal_naive_forecast = validation_data[
    ["month", "total_sales"]
].copy()

seasonal_naive_forecast = seasonal_naive_forecast.rename(
    columns={
        "total_sales": "actual_sales"
    }
)

seasonal_naive_forecast["seasonal_reference_month"] = (
    seasonal_naive_forecast["month"]
    - pd.DateOffset(years=1)
)

seasonal_naive_forecast = seasonal_naive_forecast.merge(
    train_data[
        ["month", "total_sales"]
    ].rename(
        columns={
            "month": "seasonal_reference_month",
            "total_sales": "seasonal_naive_forecast"
        }
    ),
    on="seasonal_reference_month",
    how="left",
    validate="one_to_one"
)

seasonal_naive_forecast

,month,actual_sales,seasonal_reference_month,seasonal_naive_forecast
0,2017-05-01,2.691185e+07,2016-05-01,2.477943e+07
1,2017-06-01,2.568282e+07,2016-06-01,2.220922e+07
2,2017-07-01,2.701148e+07,2016-07-01,2.346267e+07


### Step 3.10.4 — Baseline Forecast QA

In [84]:
baseline_qa = {
    "naive_forecast_rows": len(naive_forecast),

    "seasonal_naive_rows": len(seasonal_naive_forecast),

    "naive_forecast_missing": (
        naive_forecast["naive_forecast"].isna().sum()
    ),

    "seasonal_forecast_missing": (
        seasonal_naive_forecast[
            "seasonal_naive_forecast"
        ].isna().sum()
    ),

    "actual_sales_missing": (
        naive_forecast["actual_sales"].isna().sum()
    ),

    "forecast_months_match": (
    naive_forecast["month"].reset_index(drop=True).equals(
        seasonal_naive_forecast["month"].reset_index(drop=True)
    )
),

    "seasonal_reference_within_training": (
        seasonal_naive_forecast[
            "seasonal_reference_month"
        ].max()
        <= train_data["month"].max()
    )
}

baseline_qa

{'naive_forecast_rows': 3,
 'seasonal_naive_rows': 3,
 'naive_forecast_missing': np.int64(0),
 'seasonal_forecast_missing': np.int64(0),
 'actual_sales_missing': np.int64(0),
 'forecast_months_match': True,
 'seasonal_reference_within_training': True}

In [86]:
print("Naive month index:")
print(naive_forecast["month"].index)

print("\nSeasonal-naive month index:")
print(seasonal_naive_forecast["month"].index)

print("\nNaive forecast months:")
print(naive_forecast["month"].tolist())

print("\nSeasonal-naive forecast months:")
print(seasonal_naive_forecast["month"].tolist())

Naive month index:
RangeIndex(start=0, stop=3, step=1)

Seasonal-naive month index:
RangeIndex(start=0, stop=3, step=1)

Naive forecast months:
[Timestamp('2017-05-01 00:00:00'), Timestamp('2017-06-01 00:00:00'), Timestamp('2017-07-01 00:00:00')]

Seasonal-naive forecast months:
[Timestamp('2017-05-01 00:00:00'), Timestamp('2017-06-01 00:00:00'), Timestamp('2017-07-01 00:00:00')]


## Step 3.10.5 — Diagnose and correct the QA
### 1. Inspect both indexes and date columns

In [14]:
print("Naive index:")
print(naive_forecast.index)

print("\nSeasonal naive index:")
print(seasonal_naive_forecast.index)

print("\nNaive dates:")
print(naive_forecast["month"].tolist())

print("\nSeasonal naive dates:")
print(seasonal_naive_forecast["month"].tolist())

Naive index:
RangeIndex(start=52, stop=55, step=1)

Seasonal naive index:
RangeIndex(start=0, stop=3, step=1)

Naive dates:
[Timestamp('2017-05-01 00:00:00'), Timestamp('2017-06-01 00:00:00'), Timestamp('2017-07-01 00:00:00')]

Seasonal naive dates:
[Timestamp('2017-05-01 00:00:00'), Timestamp('2017-06-01 00:00:00'), Timestamp('2017-07-01 00:00:00')]


### 2. Compare the actual dates

In [15]:
forecast_months_match = (
    naive_forecast["month"].reset_index(drop=True)
    .equals(
        seasonal_naive_forecast["month"].reset_index(drop=True)
    )
)

forecast_months_match

True

### 3. Standardize both forecast indexes

In [16]:
naive_forecast = naive_forecast.reset_index(drop=True)

seasonal_naive_forecast = (
    seasonal_naive_forecast.reset_index(drop=True)
)

### 4. Rerun the QA

In [17]:
baseline_qa = {
    "naive_forecast_rows": len(naive_forecast),

    "seasonal_naive_rows": len(seasonal_naive_forecast),

    "naive_forecast_missing": (
        naive_forecast["naive_forecast"].isna().sum()
    ),

    "seasonal_forecast_missing": (
        seasonal_naive_forecast[
            "seasonal_naive_forecast"
        ].isna().sum()
    ),

    "actual_sales_missing": (
        naive_forecast["actual_sales"].isna().sum()
    ),

    "forecast_months_match": (
        naive_forecast["month"].equals(
            seasonal_naive_forecast["month"]
        )
    ),

    "seasonal_reference_within_training": (
        seasonal_naive_forecast[
            "seasonal_reference_month"
        ].max()
        <= train_data["month"].max()
    )
}

baseline_qa

{'naive_forecast_rows': 3,
 'seasonal_naive_rows': 3,
 'naive_forecast_missing': np.int64(0),
 'seasonal_forecast_missing': np.int64(0),
 'actual_sales_missing': np.int64(0),
 'forecast_months_match': True,
 'seasonal_reference_within_training': True}

### Step 3.10.6 — Calculate Forecast Evaluation Metrics

### Step 3.10.6A — Prepare a common evaluation dataset

In [18]:
forecast_comparison = (
    naive_forecast[
        ["month", "actual_sales", "naive_forecast"]
    ]
    .merge(
        seasonal_naive_forecast[
            ["month", "seasonal_naive_forecast"]
        ],
        on="month",
        how="inner",
        validate="one_to_one"
    )
)

forecast_comparison

,month,actual_sales,naive_forecast,seasonal_naive_forecast
0,2017-05-01,2.691185e+07,2.589531e+07,2.477943e+07
1,2017-06-01,2.568282e+07,2.589531e+07,2.220922e+07
2,2017-07-01,2.701148e+07,2.589531e+07,2.346267e+07


### Step 3.10.6B — Calculate individual forecast errors

In [19]:
forecast_comparison["naive_error"] = (
    forecast_comparison["naive_forecast"]
    - forecast_comparison["actual_sales"]
)

forecast_comparison["seasonal_naive_error"] = (
    forecast_comparison["seasonal_naive_forecast"]
    - forecast_comparison["actual_sales"]
)

forecast_comparison["naive_absolute_error"] = (
    forecast_comparison["naive_error"].abs()
)

forecast_comparison["seasonal_naive_absolute_error"] = (
    forecast_comparison["seasonal_naive_error"].abs()
)

forecast_comparison

,month,actual_sales,naive_forecast,seasonal_naive_forecast,naive_error,seasonal_naive_error,naive_absolute_error,seasonal_naive_absolute_error
0,2017-05-01,2.691185e+07,2.589531e+07,2.477943e+07,-1.016539e+06,-2.132415e+06,1.016539e+06,2.132415e+06
1,2017-06-01,2.568282e+07,2.589531e+07,2.220922e+07,2.124864e+05,-3.473604e+06,2.124864e+05,3.473604e+06
2,2017-07-01,2.701148e+07,2.589531e+07,2.346267e+07,-1.116169e+06,-3.548805e+06,1.116169e+06,3.548805e+06


In [20]:
import numpy as np
import pandas as pd

actual = forecast_comparison["actual_sales"]

naive_error = forecast_comparison["naive_error"]
seasonal_error = forecast_comparison["seasonal_naive_error"]

# MAE
naive_mae = naive_error.abs().mean()
seasonal_mae = seasonal_error.abs().mean()

# RMSE
naive_rmse = np.sqrt((naive_error ** 2).mean())
seasonal_rmse = np.sqrt((seasonal_error ** 2).mean())

# WAPE (%)
naive_wape = (
    naive_error.abs().sum() / actual.sum()
) * 100

seasonal_wape = (
    seasonal_error.abs().sum() / actual.sum()
) * 100

# Bias (%)
naive_bias = (
    naive_error.sum() / actual.sum()
) * 100

seasonal_bias = (
    seasonal_error.sum() / actual.sum()
) * 100

# Evaluation summary
baseline_metrics = pd.DataFrame({
    "model": ["Naive", "Seasonal Naive"],
    "MAE": [naive_mae, seasonal_mae],
    "RMSE": [naive_rmse, seasonal_rmse],
    "WAPE_pct": [naive_wape, seasonal_wape],
    "Bias_pct": [naive_bias, seasonal_bias]
})

baseline_metrics.round(2)

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,Naive,781731.40,880214.92,2.95,-2.41
1,Seasonal Naive,3051607.89,3120210.10,11.50,-11.50


### Baseline Forecast Evaluation — Initial Holdout

**Evaluation period:** May–July 2017
**Forecast horizon:** 3 months
**Target:** Monthly total retail Sales

Two baseline forecasting methods were evaluated:

* Naive Forecast: repeats the latest training-period monthly Sales value.
* Seasonal Naive Forecast: uses Sales from the corresponding month in the previous year.

#### Results

| Metric |      Naive | Seasonal Naive |
| ------ | ---------: | -------------: |
| MAE    | 781,731.40 |   3,051,607.89 |
| RMSE   | 880,214.92 |   3,120,210.10 |
| WAPE   |      2.95% |         11.50% |
| Bias   |     -2.41% |        -11.50% |

#### Initial finding

The Naive model outperformed the Seasonal Naive model across all four evaluation metrics on the initial three-month holdout.

The Naive model achieved a WAPE of 2.95%, compared with 11.50% for the Seasonal Naive model, representing an approximately 74.35% relative reduction in WAPE.

Both models exhibited negative bias, indicating net underforecasting during the validation period. The Seasonal Naive model had the larger underforecasting bias.

#### Business interpretation

The latest observed monthly Sales level provided a more accurate reference for this particular validation period than the corresponding months from the previous year.

This result is specific to the selected holdout and should not be generalized without additional chronological validation.

#### Limitations

* Only three validation observations were evaluated.
* The evaluation does not establish performance across different forecast origins.
* No causal conclusion has been established regarding why the seasonal baseline underperformed.

**Next evaluation requirement:** Conduct rolling-origin validation to assess whether the observed baseline performance is consistent across multiple chronological validation windows.


### Step 3.11.1 — First, define the rolling-validation design

## Phase 3.11 — Rolling-Origin Validation

### Objective

Evaluate whether the baseline forecasting results observed in the initial
May–July 2017 holdout are consistent across multiple historical forecast origins.

### Validation design

- Forecast target: Monthly total Sales
- Forecast grain: Month × Total Retail Business
- Forecast horizon: 3 months
- Validation method: Rolling-origin chronological validation
- Models:
  - Naive
  - Seasonal Naive
- Metrics:
  - MAE
  - RMSE
  - WAPE
  - Bias

### Chronological principle

Each validation window must occur strictly after its corresponding training
period. No future observations may be used to construct a forecast.

### Purpose

Rolling-origin validation will allow model performance to be compared across
multiple historical periods and will provide stronger evidence about whether
the Naive model consistently outperforms the Seasonal Naive model.

### Important limitation

The initial May–July 2017 holdout remains a separate final-style holdout.
Rolling-origin validation should not overwrite or replace that original
evaluation.

### Step 3.11.2 — Define Rolling Forecast Origins

#### Step 3.11.2A — Check our monthly time series

In [21]:
monthly_ts["total_sales"]

0     1.032762e+07
1     9.658960e+06
2     1.142850e+07
3     1.099346e+07
4     1.159770e+07
5     1.168934e+07
6     1.125740e+07
7     1.173779e+07
8     1.179293e+07
9     1.177562e+07
10    1.235656e+07
11    1.580312e+07
12    1.891164e+07
13    1.203835e+07
14    2.036558e+07
15    1.286125e+07
16    1.337979e+07
17    1.331996e+07
18    1.942189e+07
19    1.388518e+07
20    2.002242e+07
21    2.039610e+07
22    2.053164e+07
23    2.434045e+07
24    1.489692e+07
25    1.374240e+07
26    1.559861e+07
27    1.495507e+07
28    1.773037e+07
29    2.161536e+07
30    2.220962e+07
31    2.296367e+07
32    2.324088e+07
33    2.387827e+07
34    2.280495e+07
35    2.724398e+07
36    2.397780e+07
37    2.194741e+07
38    2.313178e+07
39    2.596303e+07
40    2.477943e+07
41    2.220922e+07
42    2.346267e+07
43    2.245241e+07
44    2.241745e+07
45    2.403039e+07
46    2.464264e+07
47    2.964029e+07
48    2.632816e+07
49    2.325011e+07
50    2.670402e+07
51    2.589531e+07
52    2.6911

In [22]:
print("Number of observations:", len(monthly_ts))
print("Start:", monthly_ts["month"].min())
print("End:", monthly_ts["month"].max())

Number of observations: 55
Start: 2013-01-01 00:00:00
End: 2017-07-01 00:00:00


In [23]:
monthly_ts[["month", "total_sales"]].head()

,month,total_sales
0,2013-01-01,1.032762e+07
1,2013-02-01,9.658960e+06
2,2013-03-01,1.142850e+07
3,2013-04-01,1.099346e+07
4,2013-05-01,1.159770e+07


In [24]:
monthly_ts[["month", "total_sales"]].tail()

,month,total_sales
50,2017-03-01,2.670402e+07
51,2017-04-01,2.589531e+07
52,2017-05-01,2.691185e+07
53,2017-06-01,2.568282e+07
54,2017-07-01,2.701148e+07


### Step 3.11.2A — Define validation parameters

In [25]:
forecast_horizon = 3
minimum_training_months = 36

print("Forecast horizon:", forecast_horizon, "months")
print("Minimum training history:", minimum_training_months, "months")

Forecast horizon: 3 months
Minimum training history: 36 months


### Step 3.11.2B — Generate the rolling windows

In [26]:
rolling_windows = []

n_observations = len(monthly_ts)

for train_end in range(
    minimum_training_months,
    n_observations - forecast_horizon + 1
):
    validation_start = train_end
    validation_end = train_end + forecast_horizon

    train_data = monthly_ts.iloc[:train_end].copy()
    validation_data = monthly_ts.iloc[
        validation_start:validation_end
    ].copy()

    rolling_windows.append({
        "train_start": train_data["month"].min(),
        "train_end": train_data["month"].max(),
        "validation_start": validation_data["month"].min(),
        "validation_end": validation_data["month"].max(),
        "train_rows": len(train_data),
        "validation_rows": len(validation_data)
    })

rolling_windows_df = pd.DataFrame(rolling_windows)

rolling_windows_df

,train_start,train_end,validation_start,validation_end,train_rows,validation_rows
0,2013-01-01,2015-12-01,2016-01-01,2016-03-01,36,3
1,2013-01-01,2016-01-01,2016-02-01,2016-04-01,37,3
2,2013-01-01,2016-02-01,2016-03-01,2016-05-01,38,3
3,2013-01-01,2016-03-01,2016-04-01,2016-06-01,39,3
4,2013-01-01,2016-04-01,2016-05-01,2016-07-01,40,3
5,2013-01-01,2016-05-01,2016-06-01,2016-08-01,41,3
6,2013-01-01,2016-06-01,2016-07-01,2016-09-01,42,3
7,2013-01-01,2016-07-01,2016-08-01,2016-10-01,43,3
8,2013-01-01,2016-08-01,2016-09-01,2016-11-01,44,3
9,2013-01-01,2016-09-01,2016-10-01,2016-12-01,45,3


### Step 3.11.2C — QA the windows

In [27]:
rolling_window_qa = {
    "window_count": len(rolling_windows_df),

    "all_training_lengths_valid": (
        rolling_windows_df["train_rows"] >= minimum_training_months
    ).all(),

    "all_validation_lengths_valid": (
        rolling_windows_df["validation_rows"] == forecast_horizon
    ).all(),

    "all_training_before_validation": (
        rolling_windows_df["train_end"]
        < rolling_windows_df["validation_start"]
    ).all(),

    "chronological_validation_windows": (
        rolling_windows_df["validation_start"]
        .is_monotonic_increasing
    )
}

rolling_window_qa

{'window_count': 17,
 'all_training_lengths_valid': np.True_,
 'all_validation_lengths_valid': np.True_,
 'all_training_before_validation': np.True_,
 'chronological_validation_windows': True}

#### Step 3.11.2D — Validate the windows

In [28]:
rolling_window_qa = {
    "window_count": len(rolling_windows_df),
    "all_training_lengths_valid": (
        rolling_windows_df["train_rows"] >= minimum_training_months
    ).all(),
    "all_validation_lengths_valid": (
        rolling_windows_df["validation_rows"] == forecast_horizon
    ).all(),
    "all_training_before_validation": (
        rolling_windows_df["train_end"]
        < rolling_windows_df["validation_start"]
    ).all(),
    "chronological_validation_windows": (
        rolling_windows_df["validation_start"].is_monotonic_increasing
    )
}

rolling_window_qa

{'window_count': 17,
 'all_training_lengths_valid': np.True_,
 'all_validation_lengths_valid': np.True_,
 'all_training_before_validation': np.True_,
 'chronological_validation_windows': True}

### Step 3.11.3A — Build the rolling forecasts

In [29]:
rolling_forecasts = []

for window_id, window in rolling_windows_df.iterrows():

    train_end = window["train_end"]
    validation_start = window["validation_start"]
    validation_end = window["validation_end"]

    # Training data
    train_data = monthly_ts[
        monthly_ts["month"] <= train_end
    ].copy()

    # Validation data
    validation_data = monthly_ts[
        (monthly_ts["month"] >= validation_start)
        & (monthly_ts["month"] <= validation_end)
    ].copy()

    # -------------------------
    # Naive forecast
    # -------------------------
    naive_value = train_data["total_sales"].iloc[-1]

    validation_data["naive_forecast"] = naive_value

    # -------------------------
    # Seasonal Naive forecast
    # -------------------------
    seasonal_reference_dates = (
        validation_data["month"]
        - pd.DateOffset(years=1)
    )

    seasonal_reference = (
        monthly_ts[
            monthly_ts["month"].isin(seasonal_reference_dates)
        ][["month", "total_sales"]]
        .rename(
            columns={
                "month": "seasonal_reference_month",
                "total_sales": "seasonal_naive_forecast"
            }
        )
    )

    validation_data = validation_data.copy()

    validation_data["seasonal_reference_month"] = (
        seasonal_reference_dates.values
    )

    validation_data = validation_data.merge(
        seasonal_reference,
        left_on="seasonal_reference_month",
        right_on="seasonal_reference_month",
        how="left",
        validate="one_to_one"
    )

    # Add window information
    validation_data["window_id"] = window_id + 1
    validation_data["train_end"] = train_end

    rolling_forecasts.append(
        validation_data[
            [
                "window_id",
                "train_end",
                "month",
                "total_sales",
                "naive_forecast",
                "seasonal_reference_month",
                "seasonal_naive_forecast"
            ]
        ]
    )

rolling_forecasts_df = pd.concat(
    rolling_forecasts,
    ignore_index=True
)

rolling_forecasts_df.head(10)

,window_id,train_end,month,total_sales,naive_forecast,seasonal_reference_month,seasonal_naive_forecast
0,1,2015-12-01,2016-01-01,2.397780e+07,2.724398e+07,2015-01-01,1.489692e+07
1,1,2015-12-01,2016-02-01,2.194741e+07,2.724398e+07,2015-02-01,1.374240e+07
2,1,2015-12-01,2016-03-01,2.313178e+07,2.724398e+07,2015-03-01,1.559861e+07
3,2,2016-01-01,2016-02-01,2.194741e+07,2.397780e+07,2015-02-01,1.374240e+07
4,2,2016-01-01,2016-03-01,2.313178e+07,2.397780e+07,2015-03-01,1.559861e+07
5,2,2016-01-01,2016-04-01,2.596303e+07,2.397780e+07,2015-04-01,1.495507e+07
6,3,2016-02-01,2016-03-01,2.313178e+07,2.194741e+07,2015-03-01,1.559861e+07
7,3,2016-02-01,2016-04-01,2.596303e+07,2.194741e+07,2015-04-01,1.495507e+07
8,3,2016-02-01,2016-05-01,2.477943e+07,2.194741e+07,2015-05-01,1.773037e+07
9,4,2016-03-01,2016-04-01,2.596303e+07,2.313178e+07,2015-04-01,1.495507e+07


### Step 3.11.3B — Basic QA

In [30]:
rolling_forecast_qa = {
    "forecast_rows": len(rolling_forecasts_df),

    "expected_rows": (
        len(rolling_windows_df) * forecast_horizon
    ),

    "naive_missing": (
        rolling_forecasts_df["naive_forecast"].isna().sum()
    ),

    "seasonal_naive_missing": (
        rolling_forecasts_df["seasonal_naive_forecast"].isna().sum()
    ),

    "actual_sales_missing": (
        rolling_forecasts_df["total_sales"].isna().sum()
    ),

    "seasonal_reference_missing": (
        rolling_forecasts_df["seasonal_reference_month"].isna().sum()
    ),

    "forecast_month_missing": (
        rolling_forecasts_df["month"].isna().sum()
    )
}

rolling_forecast_qa

{'forecast_rows': 51,
 'expected_rows': 51,
 'naive_missing': np.int64(0),
 'seasonal_naive_missing': np.int64(0),
 'actual_sales_missing': np.int64(0),
 'seasonal_reference_missing': np.int64(0),
 'forecast_month_missing': np.int64(0)}

### Step 3.11.3C — Check the first rolling window manually

In [31]:
rolling_forecasts_df[
    rolling_forecasts_df["window_id"] == 1
]

,window_id,train_end,month,total_sales,naive_forecast,seasonal_reference_month,seasonal_naive_forecast
0,1,2015-12-01,2016-01-01,2.397780e+07,2.724398e+07,2015-01-01,1.489692e+07
1,1,2015-12-01,2016-02-01,2.194741e+07,2.724398e+07,2015-02-01,1.374240e+07
2,1,2015-12-01,2016-03-01,2.313178e+07,2.724398e+07,2015-03-01,1.559861e+07


### Step 3.11.4 — Calculate Rolling-Origin Errors

In [32]:
rolling_forecasts_df["naive_error"] = (
    rolling_forecasts_df["naive_forecast"]
    - rolling_forecasts_df["total_sales"]
)

rolling_forecasts_df["seasonal_naive_error"] = (
    rolling_forecasts_df["seasonal_naive_forecast"]
    - rolling_forecasts_df["total_sales"]
)

rolling_forecasts_df["naive_absolute_error"] = (
    rolling_forecasts_df["naive_error"].abs()
)

rolling_forecasts_df["seasonal_naive_absolute_error"] = (
    rolling_forecasts_df["seasonal_naive_error"].abs()
)

rolling_forecasts_df.head()

,window_id,train_end,month,total_sales,naive_forecast,seasonal_reference_month,seasonal_naive_forecast,naive_error,seasonal_naive_error,naive_absolute_error,seasonal_naive_absolute_error
0,1,2015-12-01,2016-01-01,2.397780e+07,2.724398e+07,2015-01-01,1.489692e+07,3.266177e+06,-9.080883e+06,3.266177e+06,9.080883e+06
1,1,2015-12-01,2016-02-01,2.194741e+07,2.724398e+07,2015-02-01,1.374240e+07,5.296573e+06,-8.205013e+06,5.296573e+06,8.205013e+06
2,1,2015-12-01,2016-03-01,2.313178e+07,2.724398e+07,2015-03-01,1.559861e+07,4.112201e+06,-7.533173e+06,4.112201e+06,7.533173e+06
3,2,2016-01-01,2016-02-01,2.194741e+07,2.397780e+07,2015-02-01,1.374240e+07,2.030396e+06,-8.205013e+06,2.030396e+06,8.205013e+06
4,2,2016-01-01,2016-03-01,2.313178e+07,2.397780e+07,2015-03-01,1.559861e+07,8.460240e+05,-7.533173e+06,8.460240e+05,7.533173e+06


### Step 3.11.4A — Add a simple error QA

In [33]:
rolling_error_qa = {
    "naive_error_missing": (
        rolling_forecasts_df["naive_error"].isna().sum()
    ),

    "seasonal_error_missing": (
        rolling_forecasts_df["seasonal_naive_error"].isna().sum()
    ),

    "naive_absolute_error_missing": (
        rolling_forecasts_df["naive_absolute_error"].isna().sum()
    ),

    "seasonal_absolute_error_missing": (
        rolling_forecasts_df["seasonal_naive_absolute_error"].isna().sum()
    ),

    "negative_actual_sales": (
        (rolling_forecasts_df["total_sales"] < 0).sum()
    )
}

rolling_error_qa

{'naive_error_missing': np.int64(0),
 'seasonal_error_missing': np.int64(0),
 'naive_absolute_error_missing': np.int64(0),
 'seasonal_absolute_error_missing': np.int64(0),
 'negative_actual_sales': np.int64(0)}

### Step 3.11.5 — Calculate Metrics for Each Rolling Window

In [34]:
rolling_metrics = []

for window_id, group in rolling_forecasts_df.groupby("window_id"):

    actual = group["total_sales"]

    naive_error = group["naive_error"]
    seasonal_error = group["seasonal_naive_error"]

    rolling_metrics.append({
        "window_id": window_id,
        
        "train_end": group["train_end"].iloc[0],
        
        "validation_start": group["month"].min(),
        "validation_end": group["month"].max(),

        # Naive
        "naive_mae": naive_error.abs().mean(),
        "naive_rmse": np.sqrt((naive_error ** 2).mean()),
        "naive_wape_pct": (
            naive_error.abs().sum() / actual.sum()
        ) * 100,
        "naive_bias_pct": (
            naive_error.sum() / actual.sum()
        ) * 100,

        # Seasonal Naive
        "seasonal_naive_mae": seasonal_error.abs().mean(),
        "seasonal_naive_rmse": np.sqrt(
            (seasonal_error ** 2).mean()
        ),
        "seasonal_naive_wape_pct": (
            seasonal_error.abs().sum() / actual.sum()
        ) * 100,
        "seasonal_naive_bias_pct": (
            seasonal_error.sum() / actual.sum()
        ) * 100
    })

rolling_metrics_df = pd.DataFrame(rolling_metrics)

rolling_metrics_df

,window_id,train_end,validation_start,validation_end,naive_mae,naive_rmse,naive_wape_pct,naive_bias_pct,seasonal_naive_mae,seasonal_naive_rmse,seasonal_naive_wape_pct,seasonal_naive_bias_pct
0,1,2015-12-01,2016-01-01,2016-03-01,4.224983e+06,4.306266e+06,18.354333,18.354333,8.273023e+06,8.297256e+06,35.939979,-35.939979
1,2,2016-01-01,2016-02-01,2016-04-01,1.620547e+06,1.710688e+06,6.843312,1.254465,8.915381e+06,9.041498e+06,37.648240,-37.648240
2,3,2016-02-01,2016-03-01,2016-05-01,2.677337e+06,2.918234e+06,10.872548,-10.872548,8.530065e+06,8.710398e+06,34.640214,-34.640214
3,4,2016-03-01,2016-04-01,2016-06-01,1.800486e+06,1.964843e+06,7.404159,-4.874917,6.216960e+06,7.554625e+06,25.566074,-25.566074
4,5,2016-04-01,2016-05-01,2016-07-01,2.479251e+06,2.692194e+06,10.557292,10.557292,2.965325e+06,4.147775e+06,12.627124,-12.627124
5,6,2016-05-01,2016-06-01,2016-08-01,2.071330e+06,2.141251e+06,9.121548,9.121548,7.860572e+05,8.532677e+05,3.461572,-1.960610
6,7,2016-06-01,2016-07-01,2016-09-01,5.682930e+05,7.469159e+05,2.494974,-2.494974,8.625826e+05,9.146172e+05,3.786992,0.119476
7,8,2016-07-01,2016-08-01,2016-10-01,8.743999e+05,9.010028e+05,3.807243,2.159303,4.956055e+05,5.664427e+05,2.157926,1.716354
8,9,2016-08-01,2016-09-01,2016-11-01,1.267723e+06,1.558666e+06,5.349757,-5.251386,9.377478e+05,1.165944e+06,3.957272,-1.640690
9,10,2016-09-01,2016-10-01,2016-12-01,3.686991e+06,4.461781e+06,14.124000,-14.124000,1.462038e+06,1.745711e+06,5.600727,-5.600727


### Step 3.11.7 — Calculate overall rolling-validation metrics

In [35]:
all_actual = rolling_forecasts_df["total_sales"]

all_naive_error = rolling_forecasts_df["naive_error"]
all_seasonal_error = rolling_forecasts_df["seasonal_naive_error"]

overall_rolling_metrics = pd.DataFrame({
    "model": [
        "Naive",
        "Seasonal Naive"
    ],

    "MAE": [
        all_naive_error.abs().mean(),
        all_seasonal_error.abs().mean()
    ],

    "RMSE": [
        np.sqrt((all_naive_error ** 2).mean()),
        np.sqrt((all_seasonal_error ** 2).mean())
    ],

    "WAPE_pct": [
        (
            all_naive_error.abs().sum()
            / all_actual.sum()
        ) * 100,

        (
            all_seasonal_error.abs().sum()
            / all_actual.sum()
        ) * 100
    ],

    "Bias_pct": [
        (
            all_naive_error.sum()
            / all_actual.sum()
        ) * 100,

        (
            all_seasonal_error.sum()
            / all_actual.sum()
        ) * 100
    ]
})

overall_rolling_metrics.round(2)

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,Naive,2178014.34,2735640.76,8.80,-0.59
1,Seasonal Naive,3210528.38,4529398.49,12.98,-12.31


In [36]:
window_wins = pd.DataFrame({
    "window_id": rolling_metrics_df["window_id"],

    "naive_wape": rolling_metrics_df["naive_wape_pct"],

    "seasonal_naive_wape": (
        rolling_metrics_df["seasonal_naive_wape_pct"]
    )
})

window_wins["wape_winner"] = np.where(
    window_wins["naive_wape"]
    < window_wins["seasonal_naive_wape"],
    "Naive",
    "Seasonal Naive"
)

window_wins["wape_winner"].value_counts()

wape_winner
Naive             9
Seasonal Naive    8
Name: count, dtype: int64

# Phase 3.11 — Final baseline conclusion
## Rolling-Origin Baseline Assessment

Rolling-origin validation was performed using 17 chronological validation
windows, each containing a 3-month forecast horizon. This produced 51
out-of-sample monthly forecast observations.

### Overall rolling-validation results

| Metric | Naive | Seasonal Naive |
|---|---:|---:|
| MAE | 2,178,014.34 | 3,210,528.38 |
| RMSE | 2,735,640.76 | 4,529,398.49 |
| WAPE | 8.80% | 12.98% |
| Bias | -0.59% | -12.31% |

The Naive model produced lower MAE, RMSE, and WAPE and had substantially
lower overall bias.

The Naive model achieved the lower WAPE in 9 of 17 validation windows,
while Seasonal Naive achieved the lower WAPE in 8 of 17 windows.

### Baseline conclusion

The Naive model is the stronger overall baseline based on aggregate
rolling-origin performance. However, its advantage is not consistent across
all validation windows. Seasonal Naive outperformed Naive in 8 of the
17 windows, indicating that seasonal patterns can provide predictive value
during some periods.

The substantially negative Bias of the Seasonal Naive model (-12.31%)
indicates systematic overall underforecasting across the rolling-validation
sample. The Naive model's overall Bias (-0.59%) was much closer to zero.

The results support using the Naive model as the primary baseline for
subsequent model comparisons while retaining Seasonal Naive as an important
benchmark.

### Methodological implication

Baseline models establish the minimum performance that a more advanced
forecasting approach should beat. Future forecasting models should be
evaluated using the same chronological validation framework and the same
evaluation metrics.

### Limitation

Rolling-origin validation provides stronger evidence than the single
May–July 2017 holdout, but the evaluation still covers a relatively small
monthly history. Forecast performance may vary across different business
conditions and seasonal periods.

Further analysis should investigate seasonality, promotions, calendar
effects, and other retail drivers before selecting a production-oriented
forecasting approach.

## Phase 3.12 — Baseline Forecast Error Pattern Analysis

### Objective

Analyze the forecast errors produced by the Naive and Seasonal Naive
baseline models during rolling-origin validation.

The analysis will identify:

- periods with the largest absolute forecast errors
- underforecasting and overforecasting patterns
- calendar-month error patterns
- periods where Seasonal Naive outperformed Naive
- potential business implications for retail planning

The analysis will distinguish observed error patterns from possible
explanations. No causal conclusion will be made without supporting evidence.

In [37]:
# Phase 3.12.1 — Largest Naive Forecast Errors

naive_largest_errors = (
    rolling_forecasts_df[
        [
            "window_id",
            "train_end",
            "month",
            "total_sales",
            "naive_forecast",
            "naive_error",
            "naive_absolute_error"
        ]
    ]
    .sort_values("naive_absolute_error", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

naive_largest_errors

,window_id,train_end,month,total_sales,naive_forecast,naive_error,naive_absolute_error
0,10,2016-09-01,2016-12-01,2.964029e+07,2.241745e+07,-7.222839e+06,7.222839e+06
1,13,2016-12-01,2017-02-01,2.325011e+07,2.964029e+07,6.390175e+06,6.390175e+06
2,11,2016-10-01,2016-12-01,2.964029e+07,2.403039e+07,-5.609898e+06,5.609898e+06
3,1,2015-12-01,2016-02-01,2.194741e+07,2.724398e+07,5.296573e+06,5.296573e+06
4,12,2016-11-01,2016-12-01,2.964029e+07,2.464264e+07,-4.997647e+06,4.997647e+06
5,1,2015-12-01,2016-03-01,2.313178e+07,2.724398e+07,4.112201e+06,4.112201e+06
6,3,2016-02-01,2016-04-01,2.596303e+07,2.194741e+07,-4.015616e+06,4.015616e+06
7,5,2016-04-01,2016-06-01,2.220922e+07,2.596303e+07,3.753807e+06,3.753807e+06
8,15,2017-02-01,2017-05-01,2.691185e+07,2.325011e+07,-3.661735e+06,3.661735e+06
9,15,2017-02-01,2017-03-01,2.670402e+07,2.325011e+07,-3.453905e+06,3.453905e+06


In [38]:
# Phase 3.12.1 — Largest Seasonal Naive Forecast Errors

seasonal_largest_errors = (
    rolling_forecasts_df[
        [
            "window_id",
            "train_end",
            "month",
            "total_sales",
            "seasonal_naive_forecast",
            "seasonal_naive_error",
            "seasonal_naive_absolute_error"
        ]
    ]
    .sort_values("seasonal_naive_absolute_error", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

seasonal_largest_errors

,window_id,train_end,month,total_sales,seasonal_naive_forecast,seasonal_naive_error,seasonal_naive_absolute_error
0,4,2016-03-01,2016-04-01,2.596303e+07,1.495507e+07,-1.100796e+07,1.100796e+07
1,3,2016-02-01,2016-04-01,2.596303e+07,1.495507e+07,-1.100796e+07,1.100796e+07
2,2,2016-01-01,2016-04-01,2.596303e+07,1.495507e+07,-1.100796e+07,1.100796e+07
3,1,2015-12-01,2016-01-01,2.397780e+07,1.489692e+07,-9.080883e+06,9.080883e+06
4,1,2015-12-01,2016-02-01,2.194741e+07,1.374240e+07,-8.205013e+06,8.205013e+06
5,2,2016-01-01,2016-02-01,2.194741e+07,1.374240e+07,-8.205013e+06,8.205013e+06
6,1,2015-12-01,2016-03-01,2.313178e+07,1.559861e+07,-7.533173e+06,7.533173e+06
7,3,2016-02-01,2016-03-01,2.313178e+07,1.559861e+07,-7.533173e+06,7.533173e+06
8,2,2016-01-01,2016-03-01,2.313178e+07,1.559861e+07,-7.533173e+06,7.533173e+06
9,3,2016-02-01,2016-05-01,2.477943e+07,1.773037e+07,-7.049064e+06,7.049064e+06


In [39]:
# QA — Largest Error Tables

largest_error_qa = {
    "naive_rows": len(naive_largest_errors),
    "seasonal_naive_rows": len(seasonal_largest_errors),
    "naive_absolute_error_missing": naive_largest_errors[
        "naive_absolute_error"
    ].isna().sum(),
    "seasonal_absolute_error_missing": seasonal_largest_errors[
        "seasonal_naive_absolute_error"
    ].isna().sum(),
    "naive_sorted_descending": naive_largest_errors[
        "naive_absolute_error"
    ].is_monotonic_decreasing,
    "seasonal_sorted_descending": seasonal_largest_errors[
        "seasonal_naive_absolute_error"
    ].is_monotonic_decreasing,
}

largest_error_qa

{'naive_rows': 10,
 'seasonal_naive_rows': 10,
 'naive_absolute_error_missing': np.int64(0),
 'seasonal_absolute_error_missing': np.int64(0),
 'naive_sorted_descending': True,
 'seasonal_sorted_descending': True}

## Step 3.12.2 — Forecast Direction Analysis

Forecast errors are classified using the convention:

Forecast − Actual

Therefore:

- Negative error = underforecast
- Positive error = overforecast
- Zero error = exact forecast

The objective is to determine whether either baseline model shows a
systematic tendency to underforecast or overforecast across the rolling
validation observations.

In [40]:
# Phase 3.12.2 — Forecast Direction Analysis

naive_underforecast = (
    rolling_forecasts_df["naive_error"] < 0
).sum()

naive_overforecast = (
    rolling_forecasts_df["naive_error"] > 0
).sum()

naive_exact = (
    rolling_forecasts_df["naive_error"] == 0
).sum()

seasonal_underforecast = (
    rolling_forecasts_df["seasonal_naive_error"] < 0
).sum()

seasonal_overforecast = (
    rolling_forecasts_df["seasonal_naive_error"] > 0
).sum()

seasonal_exact = (
    rolling_forecasts_df["seasonal_naive_error"] == 0
).sum()

direction_summary = pd.DataFrame({
    "model": [
        "Naive",
        "Seasonal Naive"
    ],
    "underforecast_count": [
        naive_underforecast,
        seasonal_underforecast
    ],
    "overforecast_count": [
        naive_overforecast,
        seasonal_overforecast
    ],
    "exact_count": [
        naive_exact,
        seasonal_exact
    ]
})

direction_summary

,model,underforecast_count,overforecast_count,exact_count
0,Naive,27,24,0
1,Seasonal Naive,42,9,0


In [41]:
# QA — Forecast Direction Counts

direction_qa = {
    "naive_total": (
        naive_underforecast
        + naive_overforecast
        + naive_exact
    ),
    "seasonal_naive_total": (
        seasonal_underforecast
        + seasonal_overforecast
        + seasonal_exact
    ),
    "expected_forecast_observations": len(rolling_forecasts_df)
}

direction_qa

{'naive_total': np.int64(51),
 'seasonal_naive_total': np.int64(51),
 'expected_forecast_observations': 51}

## Step 3.12.3 — Forecast Error Patterns by Calendar Month

The rolling-origin forecast observations are grouped by calendar month to
investigate whether forecast errors show recurring monthly patterns.

Because rolling-origin windows overlap, some calendar months are represented
by multiple forecast instances. Therefore, these results describe forecast
behavior by calendar month rather than independent monthly observations.

The analysis compares absolute error and forecast direction for the Naive and
Seasonal Naive baselines.

In [42]:
# Phase 3.12.3 — Error Patterns by Calendar Month

calendar_month_errors = (
    rolling_forecasts_df
    .assign(
        calendar_month=rolling_forecasts_df["month"].dt.month
    )
    .groupby("calendar_month")
    .agg(
        forecast_instances=("month", "size"),
        
        naive_total_absolute_error=(
            "naive_absolute_error",
            "sum"
        ),
        naive_underforecast_count=(
            "naive_error",
            lambda x: (x < 0).sum()
        ),
        naive_overforecast_count=(
            "naive_error",
            lambda x: (x > 0).sum()
        ),
        
        seasonal_naive_total_absolute_error=(
            "seasonal_naive_absolute_error",
            "sum"
        ),
        seasonal_underforecast_count=(
            "seasonal_naive_error",
            lambda x: (x < 0).sum()
        ),
        seasonal_overforecast_count=(
            "seasonal_naive_error",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

calendar_month_errors

,calendar_month,forecast_instances,naive_total_absolute_error,naive_underforecast_count,naive_overforecast_count,seasonal_naive_total_absolute_error,seasonal_underforecast_count,seasonal_overforecast_count
0,1,4,1.056159e+07,2,2,1.613195e+07,4,0
1,2,5,1.818772e+07,0,5,2.031814e+07,5,0
2,3,6,1.290863e+07,3,3,3.331623e+07,6,0
3,4,6,1.271884e+07,4,2,3.322702e+07,3,3
4,5,6,1.054937e+07,5,1,2.754444e+07,6,0
5,6,5,8.480264e+06,0,5,8.728782e+06,5,0
6,7,4,6.186735e+06,2,2,7.307965e+06,4,0
7,8,3,3.580472e+06,1,2,1.533780e+06,0,3
8,9,3,1.288420e+06,1,2,2.470303e+06,0,3
9,10,3,3.758635e+06,3,0,4.563665e+05,3,0


In [43]:
# Calculate average absolute error per forecast instance

calendar_month_errors["naive_mean_absolute_error"] = (
    calendar_month_errors["naive_total_absolute_error"]
    / calendar_month_errors["forecast_instances"]
)

calendar_month_errors["seasonal_naive_mean_absolute_error"] = (
    calendar_month_errors["seasonal_naive_total_absolute_error"]
    / calendar_month_errors["forecast_instances"]
)

calendar_month_errors

,calendar_month,forecast_instances,naive_total_absolute_error,naive_underforecast_count,naive_overforecast_count,seasonal_naive_total_absolute_error,seasonal_underforecast_count,seasonal_overforecast_count,naive_mean_absolute_error,seasonal_naive_mean_absolute_error
0,1,4,1.056159e+07,2,2,1.613195e+07,4,0,2.640399e+06,4.032987e+06
1,2,5,1.818772e+07,0,5,2.031814e+07,5,0,3.637544e+06,4.063627e+06
2,3,6,1.290863e+07,3,3,3.331623e+07,6,0,2.151438e+06,5.552705e+06
3,4,6,1.271884e+07,4,2,3.322702e+07,3,3,2.119806e+06,5.537837e+06
4,5,6,1.054937e+07,5,1,2.754444e+07,6,0,1.758228e+06,4.590740e+06
5,6,5,8.480264e+06,0,5,8.728782e+06,5,0,1.696053e+06,1.745756e+06
6,7,4,6.186735e+06,2,2,7.307965e+06,4,0,1.546684e+06,1.826991e+06
7,8,3,3.580472e+06,1,2,1.533780e+06,0,3,1.193491e+06,5.112599e+05
8,9,3,1.288420e+06,1,2,2.470303e+06,0,3,4.294734e+05,8.234344e+05
9,10,3,3.758635e+06,3,0,4.563665e+05,3,0,1.252878e+06,1.521222e+05


In [44]:
# QA — Calendar Month Error Analysis

calendar_month_qa = {
    "calendar_month_count": calendar_month_errors["calendar_month"].nunique(),
    "forecast_instances_total": calendar_month_errors[
        "forecast_instances"
    ].sum(),
    "expected_forecast_instances": len(rolling_forecasts_df),
    "naive_missing": calendar_month_errors[
        "naive_mean_absolute_error"
    ].isna().sum(),
    "seasonal_naive_missing": calendar_month_errors[
        "seasonal_naive_mean_absolute_error"
    ].isna().sum()
}

calendar_month_qa

{'calendar_month_count': 12,
 'forecast_instances_total': np.int64(51),
 'expected_forecast_instances': 51,
 'naive_missing': np.int64(0),
 'seasonal_naive_missing': np.int64(0)}

## Step 3.12.4 — Validation Windows Where Seasonal Naive Outperformed

The Seasonal Naive model was not consistently inferior to the Naive model.

Although Naive achieved better WAPE in 9 of the 17 rolling validation
windows, Seasonal Naive achieved better WAPE in 8 windows.

This analysis identifies those windows and examines the validation periods
where the prior-year seasonal reference provided a better forecast.

The purpose is to understand when seasonal information may add predictive
value rather than assuming that one baseline is universally superior.

In [45]:
# Phase 3.12.4 — Seasonal Naive Winning Windows

seasonal_wins = (
    rolling_metrics_df[
        rolling_metrics_df["seasonal_naive_wape_pct"]
        < rolling_metrics_df["naive_wape_pct"]
    ]
    [
        [
            "window_id",
            "train_end",
            "validation_start",
            "validation_end",
            "naive_wape_pct",
            "seasonal_naive_wape_pct",
            "naive_bias_pct",
            "seasonal_naive_bias_pct"
        ]
    ]
    .copy()
)

seasonal_wins["wape_improvement_pct"] = (
    (
        seasonal_wins["naive_wape_pct"]
        - seasonal_wins["seasonal_naive_wape_pct"]
    )
    / seasonal_wins["naive_wape_pct"]
    * 100
)

seasonal_wins = seasonal_wins.reset_index(drop=True)

seasonal_wins

,window_id,train_end,validation_start,validation_end,naive_wape_pct,seasonal_naive_wape_pct,naive_bias_pct,seasonal_naive_bias_pct,wape_improvement_pct
0,6,2016-05-01,2016-06-01,2016-08-01,9.121548,3.461572,9.121548,-1.960610,62.050615
1,8,2016-07-01,2016-08-01,2016-10-01,3.807243,2.157926,2.159303,1.716354,43.320504
2,9,2016-08-01,2016-09-01,2016-11-01,5.349757,3.957272,-5.251386,-1.640690,26.028941
3,10,2016-09-01,2016-10-01,2016-12-01,14.124000,5.600727,-14.124000,-5.600727,60.346031
4,11,2016-10-01,2016-11-01,2017-01-01,10.569164,8.168042,-10.569164,-8.168042,22.718181
5,12,2016-11-01,2016-12-01,2017-02-01,10.194196,7.636297,-6.678535,-7.636297,25.091716
6,13,2016-12-01,2017-01-01,2017-03-01,16.568161,9.471786,16.568161,-9.471786,42.831397
7,15,2017-02-01,2017-03-01,2017-05-01,12.276056,7.259820,-12.276056,-7.089488,40.861948


In [46]:
# QA — Seasonal Naive Winning Windows

seasonal_wins_qa = {
    "seasonal_winning_windows": len(seasonal_wins),
    "expected_seasonal_wins": 8,
    "all_wape_improvements_positive": (
        seasonal_wins["wape_improvement_pct"] > 0
    ).all()
}

seasonal_wins_qa

{'seasonal_winning_windows': 8,
 'expected_seasonal_wins': 8,
 'all_wape_improvements_positive': np.True_}

## Step 3.12.5 — Baseline Performance by Validation Window

The rolling-origin results are compared at the individual validation-window
level to identify which baseline produced the lower WAPE in each period.

This provides a more detailed view than aggregate metrics alone and helps
identify periods where the relative performance of the two baselines changed.

In [47]:
# Phase 3.12.5 — Model Winner by Validation Window

window_comparison = rolling_metrics_df[
    [
        "window_id",
        "train_end",
        "validation_start",
        "validation_end",
        "naive_wape_pct",
        "seasonal_naive_wape_pct"
    ]
].copy()

window_comparison["winner"] = np.where(
    window_comparison["naive_wape_pct"]
    < window_comparison["seasonal_naive_wape_pct"],
    "Naive",
    "Seasonal Naive"
)

window_comparison["wape_difference_pct_points"] = (
    window_comparison["naive_wape_pct"]
    - window_comparison["seasonal_naive_wape_pct"]
)

window_comparison

,window_id,train_end,validation_start,validation_end,naive_wape_pct,seasonal_naive_wape_pct,winner,wape_difference_pct_points
0,1,2015-12-01,2016-01-01,2016-03-01,18.354333,35.939979,Naive,-17.585646
1,2,2016-01-01,2016-02-01,2016-04-01,6.843312,37.648240,Naive,-30.804928
2,3,2016-02-01,2016-03-01,2016-05-01,10.872548,34.640214,Naive,-23.767666
3,4,2016-03-01,2016-04-01,2016-06-01,7.404159,25.566074,Naive,-18.161915
4,5,2016-04-01,2016-05-01,2016-07-01,10.557292,12.627124,Naive,-2.069831
5,6,2016-05-01,2016-06-01,2016-08-01,9.121548,3.461572,Seasonal Naive,5.659977
6,7,2016-06-01,2016-07-01,2016-09-01,2.494974,3.786992,Naive,-1.292018
7,8,2016-07-01,2016-08-01,2016-10-01,3.807243,2.157926,Seasonal Naive,1.649317
8,9,2016-08-01,2016-09-01,2016-11-01,5.349757,3.957272,Seasonal Naive,1.392485
9,10,2016-09-01,2016-10-01,2016-12-01,14.124000,5.600727,Seasonal Naive,8.523274


In [48]:
# QA — Window Comparison

window_comparison_qa = {
    "window_count": len(window_comparison),
    "expected_window_count": 17,
    "winner_missing": window_comparison["winner"].isna().sum(),
    "naive_wins": (
        window_comparison["winner"] == "Naive"
    ).sum(),
    "seasonal_naive_wins": (
        window_comparison["winner"] == "Seasonal Naive"
    ).sum(),
    "total_wins": window_comparison["winner"].notna().sum()
}

window_comparison_qa

{'window_count': 17,
 'expected_window_count': 17,
 'winner_missing': np.int64(0),
 'naive_wins': np.int64(9),
 'seasonal_naive_wins': np.int64(8),
 'total_wins': np.int64(17)}

## Step 3.12.6 — Model Win Magnitude

The number of validation-window wins does not show how large each model's
advantage was.

This analysis compares the WAPE difference across validation windows to
identify periods where one baseline substantially outperformed the other.

WAPE difference is calculated as:

Naive WAPE − Seasonal Naive WAPE

Therefore:

- Negative value = Naive has lower WAPE
- Positive value = Seasonal Naive has lower WAPE

In [49]:
# Phase 3.12.6 — Model Win Magnitude

naive_win_magnitude = (
    window_comparison[
        window_comparison["winner"] == "Naive"
    ]
    .assign(
        absolute_wape_advantage=lambda df:
            df["wape_difference_pct_points"].abs()
    )
    .sort_values(
        "absolute_wape_advantage",
        ascending=False
    )
    [
        [
            "window_id",
            "validation_start",
            "validation_end",
            "naive_wape_pct",
            "seasonal_naive_wape_pct",
            "wape_difference_pct_points",
            "absolute_wape_advantage"
        ]
    ]
)

seasonal_win_magnitude = (
    window_comparison[
        window_comparison["winner"] == "Seasonal Naive"
    ]
    .assign(
        absolute_wape_advantage=lambda df:
            df["wape_difference_pct_points"].abs()
    )
    .sort_values(
        "absolute_wape_advantage",
        ascending=False
    )
    [
        [
            "window_id",
            "validation_start",
            "validation_end",
            "naive_wape_pct",
            "seasonal_naive_wape_pct",
            "wape_difference_pct_points",
            "absolute_wape_advantage"
        ]
    ]
)

print("Naive wins — magnitude:")
display(naive_win_magnitude)

print("\nSeasonal Naive wins — magnitude:")
display(seasonal_win_magnitude)

Naive wins — magnitude:


,window_id,validation_start,validation_end,naive_wape_pct,seasonal_naive_wape_pct,wape_difference_pct_points,absolute_wape_advantage
1,2,2016-02-01,2016-04-01,6.843312,37.648240,-30.804928,30.804928
2,3,2016-03-01,2016-05-01,10.872548,34.640214,-23.767666,23.767666
3,4,2016-04-01,2016-06-01,7.404159,25.566074,-18.161915,18.161915
0,1,2016-01-01,2016-03-01,18.354333,35.939979,-17.585646,17.585646
16,17,2017-05-01,2017-07-01,2.945996,11.500147,-8.554150,8.554150
15,16,2017-04-01,2017-06-01,2.596171,7.228611,-4.632440,4.632440
4,5,2016-05-01,2016-07-01,10.557292,12.627124,-2.069831,2.069831
13,14,2017-02-01,2017-04-01,5.124305,6.516406,-1.392100,1.392100
6,7,2016-07-01,2016-09-01,2.494974,3.786992,-1.292018,1.292018



Seasonal Naive wins — magnitude:


,window_id,validation_start,validation_end,naive_wape_pct,seasonal_naive_wape_pct,wape_difference_pct_points,absolute_wape_advantage
9,10,2016-10-01,2016-12-01,14.124000,5.600727,8.523274,8.523274
12,13,2017-01-01,2017-03-01,16.568161,9.471786,7.096375,7.096375
5,6,2016-06-01,2016-08-01,9.121548,3.461572,5.659977,5.659977
14,15,2017-03-01,2017-05-01,12.276056,7.259820,5.016236,5.016236
11,12,2016-12-01,2017-02-01,10.194196,7.636297,2.557899,2.557899
10,11,2016-11-01,2017-01-01,10.569164,8.168042,2.401122,2.401122
7,8,2016-08-01,2016-10-01,3.807243,2.157926,1.649317,1.649317
8,9,2016-09-01,2016-11-01,5.349757,3.957272,1.392485,1.392485


In [50]:
win_magnitude_qa = {
    "naive_win_rows": len(naive_win_magnitude),
    "seasonal_win_rows": len(seasonal_win_magnitude),
    "total_win_rows": (
        len(naive_win_magnitude)
        + len(seasonal_win_magnitude)
    ),
    "expected_total_windows": 17
}

win_magnitude_qa

{'naive_win_rows': 9,
 'seasonal_win_rows': 8,
 'total_win_rows': 17,
 'expected_total_windows': 17}

## Phase 3.13 — Advanced Forecasting Model Development.

### Step 1 — Confirm the modeling dataset

In [51]:
# Phase 3.13.1 — Confirm modeling dataset

modeling_data = monthly_ts[
    [
        "month",
        "total_sales",
        "is_partial_month",
        "is_forecast_training_period"
    ]
].copy()

modeling_data.head()

,month,total_sales,is_partial_month,is_forecast_training_period
0,2013-01-01,1.032762e+07,False,True
1,2013-02-01,9.658960e+06,False,True
2,2013-03-01,1.142850e+07,False,True
3,2013-04-01,1.099346e+07,False,True
4,2013-05-01,1.159770e+07,False,True


In [52]:
modeling_data.tail()

,month,total_sales,is_partial_month,is_forecast_training_period
50,2017-03-01,2.670402e+07,False,True
51,2017-04-01,2.589531e+07,False,True
52,2017-05-01,2.691185e+07,False,True
53,2017-06-01,2.568282e+07,False,True
54,2017-07-01,2.701148e+07,False,True


In [53]:
modeling_data.shape

(55, 4)

### Step 2 — Verify the forecasting target

In [54]:
# Target validation

target_check = {
    "rows": len(modeling_data),
    "missing_sales": modeling_data["total_sales"].isna().sum(),
    "negative_sales": (modeling_data["total_sales"] < 0).sum(),
    "partial_months": modeling_data["is_partial_month"].sum(),
    "training_period_flags": modeling_data["is_forecast_training_period"].value_counts().to_dict()
}

target_check

{'rows': 55,
 'missing_sales': np.int64(0),
 'negative_sales': np.int64(0),
 'partial_months': np.int64(0),
 'training_period_flags': {True: 55}}

### Step 3 — Create the clean modeling series

In [55]:
# Create the monthly time-series used by forecasting models

forecast_series = (
    modeling_data
    .loc[
        modeling_data["is_forecast_training_period"] == True,
        ["month", "total_sales"]
    ]
    .copy()
)

forecast_series = forecast_series.set_index("month")["total_sales"]

forecast_series.head()

month
2013-01-01    1.032762e+07
2013-02-01    9.658960e+06
2013-03-01    1.142850e+07
2013-04-01    1.099346e+07
2013-05-01    1.159770e+07
Name: total_sales, dtype: float64

In [56]:
forecast_series.tail()

month
2017-03-01    2.670402e+07
2017-04-01    2.589531e+07
2017-05-01    2.691185e+07
2017-06-01    2.568282e+07
2017-07-01    2.701148e+07
Name: total_sales, dtype: float64

In [57]:
len(forecast_series)

55

### Step 4 — Final modeling readiness check

In [58]:
modeling_readiness = {
    "observations": len(forecast_series),
    "missing_values": forecast_series.isna().sum(),
    "negative_values": (forecast_series < 0).sum(),
    "start": forecast_series.index.min(),
    "end": forecast_series.index.max(),
    "chronological": forecast_series.index.is_monotonic_increasing,
    "duplicate_months": forecast_series.index.duplicated().sum()
}

modeling_readiness

{'observations': 55,
 'missing_values': np.int64(0),
 'negative_values': np.int64(0),
 'start': Timestamp('2013-01-01 00:00:00'),
 'end': Timestamp('2017-07-01 00:00:00'),
 'chronological': True,
 'duplicate_months': np.int64(0)}

### Step 1 — Create the blended forecast

In [59]:
# Phase 3.13.2 — Create a 50/50 blended baseline

blended_forecasts_df = rolling_forecasts_df.copy()

blended_forecasts_df["blended_forecast"] = (
    0.5 * blended_forecasts_df["naive_forecast"]
    + 0.5 * blended_forecasts_df["seasonal_naive_forecast"]
)

blended_forecasts_df[
    [
        "window_id",
        "month",
        "total_sales",
        "naive_forecast",
        "seasonal_naive_forecast",
        "blended_forecast"
    ]
].head(10)

,window_id,month,total_sales,naive_forecast,seasonal_naive_forecast,blended_forecast
0,1,2016-01-01,2.397780e+07,2.724398e+07,1.489692e+07,2.107045e+07
1,1,2016-02-01,2.194741e+07,2.724398e+07,1.374240e+07,2.049319e+07
2,1,2016-03-01,2.313178e+07,2.724398e+07,1.559861e+07,2.142129e+07
3,2,2016-02-01,2.194741e+07,2.397780e+07,1.374240e+07,1.886010e+07
4,2,2016-03-01,2.313178e+07,2.397780e+07,1.559861e+07,1.978821e+07
5,2,2016-04-01,2.596303e+07,2.397780e+07,1.495507e+07,1.946644e+07
6,3,2016-03-01,2.313178e+07,2.194741e+07,1.559861e+07,1.877301e+07
7,3,2016-04-01,2.596303e+07,2.194741e+07,1.495507e+07,1.845124e+07
8,3,2016-05-01,2.477943e+07,2.194741e+07,1.773037e+07,1.983889e+07
9,4,2016-04-01,2.596303e+07,2.313178e+07,1.495507e+07,1.904342e+07


### Step 2 — Validate the new forecast

In [60]:
blended_qa = {
    "rows": len(blended_forecasts_df),
    "expected_rows": 51,
    "blended_forecast_missing": (
        blended_forecasts_df["blended_forecast"].isna().sum()
    ),
    "actual_sales_missing": (
        blended_forecasts_df["total_sales"].isna().sum()
    ),
    "all_blends_between_baselines": (
        (
            blended_forecasts_df["blended_forecast"]
            >= blended_forecasts_df[
                ["naive_forecast", "seasonal_naive_forecast"]
            ].min(axis=1)
        )
        &
        (
            blended_forecasts_df["blended_forecast"]
            <= blended_forecasts_df[
                ["naive_forecast", "seasonal_naive_forecast"]
            ].max(axis=1)
        )
    ).all(),
    "all_forecasts_nonnegative": (
        blended_forecasts_df["blended_forecast"] >= 0
    ).all()
}

blended_qa

{'rows': 51,
 'expected_rows': 51,
 'blended_forecast_missing': np.int64(0),
 'actual_sales_missing': np.int64(0),
 'all_blends_between_baselines': np.True_,
 'all_forecasts_nonnegative': np.True_}

### Step 3 — Calculate forecast errors

In [61]:
blended_forecasts_df["blended_error"] = (
    blended_forecasts_df["blended_forecast"]
    - blended_forecasts_df["total_sales"]
)

blended_forecasts_df["blended_absolute_error"] = (
    blended_forecasts_df["blended_error"].abs()
)

blended_forecasts_df[
    [
        "window_id",
        "month",
        "total_sales",
        "blended_forecast",
        "blended_error",
        "blended_absolute_error"
    ]
].head()

,window_id,month,total_sales,blended_forecast,blended_error,blended_absolute_error
0,1,2016-01-01,2.397780e+07,2.107045e+07,-2.907353e+06,2.907353e+06
1,1,2016-02-01,2.194741e+07,2.049319e+07,-1.454220e+06,1.454220e+06
2,1,2016-03-01,2.313178e+07,2.142129e+07,-1.710486e+06,1.710486e+06
3,2,2016-02-01,2.194741e+07,1.886010e+07,-3.087308e+06,3.087308e+06
4,2,2016-03-01,2.313178e+07,1.978821e+07,-3.343575e+06,3.343575e+06


### Step 4 — Evaluate all three baseline models

In [62]:
# Phase 3.13.2 — Compare baseline models

import numpy as np
import pandas as pd

model_comparison_data = {
    "Naive": (
        rolling_forecasts_df["naive_forecast"]
    ),
    "Seasonal Naive": (
        rolling_forecasts_df["seasonal_naive_forecast"]
    ),
    "50/50 Blended": (
        blended_forecasts_df["blended_forecast"]
    )
}

actual = rolling_forecasts_df["total_sales"]

comparison_results = []

for model_name, forecast in model_comparison_data.items():

    # Forecast error: Forecast - Actual
    errors = forecast - actual

    # Absolute errors
    absolute_errors = errors.abs()

    # Evaluation metrics
    mae = absolute_errors.mean()

    rmse = np.sqrt((errors ** 2).mean())

    wape_pct = (
        absolute_errors.sum() / actual.abs().sum()
    ) * 100

    bias_pct = (
        errors.sum() / actual.abs().sum()
    ) * 100

    comparison_results.append({
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE_pct": wape_pct,
        "Bias_pct": bias_pct
    })

baseline_comparison_df = pd.DataFrame(comparison_results)

baseline_comparison_df.round(2)

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,Naive,2178014.34,2735640.76,8.80,-0.59
1,Seasonal Naive,3210528.38,4529398.49,12.98,-12.31
2,50/50 Blended,2067404.61,2737131.72,8.36,-6.45


### Step 5 — Validate the comparison

In [63]:
comparison_qa = {
    "model_count": len(baseline_comparison_df),
    "expected_models": 3,
    "duplicate_model_names": (
        baseline_comparison_df["model"].duplicated().sum()
    ),
    "missing_metric_values": (
        baseline_comparison_df[
            ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
        ].isna().sum().sum()
    ),
    "all_metrics_finite": np.isfinite(
        baseline_comparison_df[
            ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
        ].to_numpy()
    ).all(),
    "all_mae_nonnegative": (
        baseline_comparison_df["MAE"] >= 0
    ).all(),
    "all_rmse_nonnegative": (
        baseline_comparison_df["RMSE"] >= 0
    ).all()
}

comparison_qa

{'model_count': 3,
 'expected_models': 3,
 'duplicate_model_names': np.int64(0),
 'missing_metric_values': np.int64(0),
 'all_metrics_finite': np.True_,
 'all_mae_nonnegative': np.True_,
 'all_rmse_nonnegative': np.True_}

In [64]:
# Phase 3.13.2 — Window-level blended forecast evaluation

window_comparison = (
    blended_forecasts_df
    .groupby("window_id")
    .apply(
        lambda group: pd.Series({
            "naive_wape_pct": (
                group["naive_absolute_error"].sum()
                / group["total_sales"].abs().sum()
            ) * 100,
            "seasonal_naive_wape_pct": (
                group["seasonal_naive_absolute_error"].sum()
                / group["total_sales"].abs().sum()
            ) * 100,
            "blended_wape_pct": (
                group["blended_absolute_error"].sum()
                / group["total_sales"].abs().sum()
            ) * 100
        }),
        include_groups=False
    )
    .reset_index()
)

window_comparison["blended_vs_naive_improvement_pp"] = (
    window_comparison["naive_wape_pct"]
    - window_comparison["blended_wape_pct"]
)

window_comparison["blended_wins"] = (
    window_comparison["blended_wape_pct"]
    < window_comparison["naive_wape_pct"]
)

window_comparison.round(2)

,window_id,naive_wape_pct,seasonal_naive_wape_pct,blended_wape_pct,blended_vs_naive_improvement_pp,blended_wins
0,1,18.35,35.94,8.79,9.56,True
1,2,6.84,37.65,18.20,-11.35,False
2,3,10.87,34.64,22.76,-11.88,False
3,4,7.40,25.57,15.67,-8.27,False
4,5,10.56,12.63,7.29,3.27,True
5,6,9.12,3.46,3.58,5.54,True
6,7,2.49,3.79,2.48,0.01,True
7,8,3.81,2.16,2.98,0.82,True
8,9,5.35,3.96,4.65,0.70,True
9,10,14.12,5.60,9.86,4.26,True


In [65]:
blend_window_summary = {
    "windows_evaluated": len(window_comparison),
    "blend_wins_vs_naive": int(
        window_comparison["blended_wins"].sum()
    ),
    "blend_ties_vs_naive": int(
        (
            window_comparison["blended_wape_pct"]
            == window_comparison["naive_wape_pct"]
        ).sum()
    ),
    "blend_mean_wape": window_comparison["blended_wape_pct"].mean(),
    "naive_mean_window_wape": window_comparison["naive_wape_pct"].mean(),
    "blend_improvement_windows": int(
        (
            window_comparison["blended_vs_naive_improvement_pp"] > 0
        ).sum()
    ),
    "blend_worse_windows": int(
        (
            window_comparison["blended_vs_naive_improvement_pp"] < 0
        ).sum()
    )
}

blend_window_summary

{'windows_evaluated': 17,
 'blend_wins_vs_naive': 12,
 'blend_ties_vs_naive': 0,
 'blend_mean_wape': np.float64(8.331854416921564),
 'naive_mean_window_wape': np.float64(8.776659741259335),
 'blend_improvement_windows': 12,
 'blend_worse_windows': 5}

## Phase 3.13.2 — Blended Baseline Evaluation

### Objective

Test whether combining the Naive and Seasonal Naive forecasts improves
monthly Sales forecast accuracy.

### Method

A fixed 50/50 blend was evaluated using the same 17 rolling-origin windows
and 51 forecast instances as the existing baseline models.

Blended Forecast = 0.5 × Naive Forecast
                 + 0.5 × Seasonal Naive Forecast

### Aggregate Results

| Model | MAE | RMSE | WAPE | Bias |
|---|---:|---:|---:|---:|
| Naive | 2.18M | 2.74M | 8.80% | -0.59% |
| Seasonal Naive | 3.21M | 4.53M | 12.98% | -12.31% |
| 50/50 Blended | 2.07M | 2.74M | 8.36% | -6.45% |

### Rolling-Window Results

The blend achieved lower WAPE than Naive in 12 of 17 windows and higher
WAPE in 5 windows.

Mean window-level WAPE was 8.33% for the blend and 8.78% for Naive.
This is an unweighted average of window-level WAPE values and differs from
pooled WAPE.

The blend's performance varied considerably across windows. It improved
accuracy in several periods but performed substantially worse than Naive
in Windows 2, 3, and 4.

### Interpretation

The 50/50 blend reduced aggregate MAE by approximately 5.1% and pooled
WAPE by 0.44 percentage points compared with Naive.

However, RMSE was slightly higher, and Bias moved from -0.59% to -6.45%.
The blend therefore reduced typical absolute error while increasing the
overall underforecasting tendency.

### Decision

Retain Naive as the primary baseline and the 50/50 blend as an alternative
candidate. Do not declare the blend the overall winner based on WAPE alone.

The current results are model-development evidence, not independent final
test results. The rolling windows overlap, and the models have already
been compared on these periods.

### Next Step

Proceed to ETS model development and evaluate it using the same
chronological validation framework.

In [66]:
# Phase 3.13.3 — Check ETS availability

import statsmodels
from statsmodels.tsa.holtwinters import ExponentialSmoothing

print("statsmodels version:", statsmodels.__version__)
print("Holt-Winters ExponentialSmoothing: available")

statsmodels version: 0.15.0
Holt-Winters ExponentialSmoothing: available


In [67]:
# Phase 3.13.3 — Initial ETS specification check

from statsmodels.tsa.holtwinters import ExponentialSmoothing

ets_training = forecast_series.loc[
    forecast_series.index <= "2015-12-01"
].copy()

ets_model = ExponentialSmoothing(
    ets_training,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    damped_trend=False,
    initialization_method="estimated"
)

ets_fit = ets_model.fit(optimized=True)

ets_diagnostic_forecast = ets_fit.forecast(3)

print("Training observations:", len(ets_training))
print("Training start:", ets_training.index.min())
print("Training end:", ets_training.index.max())
print("\nThree-month diagnostic forecast:")
print(ets_diagnostic_forecast)

Training observations: 36
Training start: 2013-01-01 00:00:00
Training end: 2015-12-01 00:00:00

Three-month diagnostic forecast:
2016-01-01    2.346665e+07
2016-02-01    1.930117e+07
2016-03-01    2.421934e+07
Freq: MS, dtype: float64


d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [68]:
ets_diagnostic_qa = {
    "training_observations": len(ets_training),
    "expected_training_observations": 36,
    "forecast_observations": len(ets_diagnostic_forecast),
    "expected_forecast_observations": 3,
    "forecast_index_matches_expected_months": (
        ets_diagnostic_forecast.index.equals(
            forecast_series.loc["2016-01-01":"2016-03-01"].index
        )
    ),
    "forecast_values_finite": (
        ets_diagnostic_forecast.notna().all()
        and __import__("numpy").isfinite(
            ets_diagnostic_forecast.to_numpy()
        ).all()
    )
}

ets_diagnostic_qa

{'training_observations': 36,
 'expected_training_observations': 36,
 'forecast_observations': 3,
 'expected_forecast_observations': 3,
 'forecast_index_matches_expected_months': True,
 'forecast_values_finite': np.True_}

In [69]:
# Phase 3.13.3 — ETS rolling-origin forecasting

import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Use the same 17 windows and 51 forecast instances
ets_rolling_forecasts_df = rolling_forecasts_df[
    ["window_id", "train_end", "month", "total_sales"]
].copy()

ets_rolling_forecasts_df["ets_forecast"] = np.nan

# Fit a separate model at each forecast origin
window_origins = (
    ets_rolling_forecasts_df[
        ["window_id", "train_end"]
    ]
    .drop_duplicates()
    .sort_values("window_id")
)

for _, window in window_origins.iterrows():

    window_id = window["window_id"]
    train_end = pd.Timestamp(window["train_end"])

    # Only use observations available at this forecast origin
    training_data = forecast_series.loc[
        forecast_series.index <= train_end
    ].copy()

    # Select the three months belonging to this validation window
    forecast_mask = (
        ets_rolling_forecasts_df["window_id"] == window_id
    )

    validation_months = (
        ets_rolling_forecasts_df.loc[forecast_mask, "month"]
        .sort_values()
    )

    # Fit additive-trend, additive-seasonal ETS
    model = ExponentialSmoothing(
        training_data,
        trend="add",
        seasonal="add",
        seasonal_periods=12,
        damped_trend=False,
        initialization_method="estimated"
    )

    fitted_model = model.fit(optimized=True)

    # Forecast as many steps as the validation window requires
    predictions = fitted_model.forecast(len(validation_months))

    # Ensure forecast dates align with validation dates
    if not predictions.index.equals(
        pd.DatetimeIndex(validation_months)
    ):
        raise ValueError(
            f"Forecast month mismatch in window {window_id}"
        )

    # Store forecasts in chronological order
    row_indices = (
        ets_rolling_forecasts_df.loc[forecast_mask]
        .sort_values("month")
        .index
    )

    ets_rolling_forecasts_df.loc[
        row_indices, "ets_forecast"
    ] = predictions.to_numpy()

ets_rolling_forecasts_df.head(10)

d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: 

,window_id,train_end,month,total_sales,ets_forecast
0,1,2015-12-01,2016-01-01,2.397780e+07,2.346665e+07
1,1,2015-12-01,2016-02-01,2.194741e+07,1.930117e+07
2,1,2015-12-01,2016-03-01,2.313178e+07,2.421934e+07
3,2,2016-01-01,2016-02-01,2.194741e+07,1.936486e+07
4,2,2016-01-01,2016-03-01,2.313178e+07,2.428877e+07
5,2,2016-01-01,2016-04-01,2.596303e+07,2.006642e+07
6,3,2016-02-01,2016-03-01,2.313178e+07,2.485719e+07
7,3,2016-02-01,2016-04-01,2.596303e+07,2.062178e+07
8,3,2016-02-01,2016-05-01,2.477943e+07,2.211446e+07
9,4,2016-03-01,2016-04-01,2.596303e+07,2.026349e+07


In [70]:
ets_rolling_qa = {
    "window_count": (
        ets_rolling_forecasts_df["window_id"].nunique()
    ),
    "expected_windows": 17,
    "forecast_rows": len(ets_rolling_forecasts_df),
    "expected_forecast_rows": 51,
    "missing_ets_forecasts": (
        ets_rolling_forecasts_df["ets_forecast"].isna().sum()
    ),
    "missing_actual_sales": (
        ets_rolling_forecasts_df["total_sales"].isna().sum()
    ),
    "all_forecasts_finite": np.isfinite(
        ets_rolling_forecasts_df["ets_forecast"].to_numpy()
    ).all(),
    "all_training_origins_before_validation": (
        (
            ets_rolling_forecasts_df["train_end"]
            < ets_rolling_forecasts_df["month"]
        ).all()
    ),
    "all_forecasts_nonnegative": (
        ets_rolling_forecasts_df["ets_forecast"] >= 0
    ).all()
}

ets_rolling_qa

{'window_count': 17,
 'expected_windows': 17,
 'forecast_rows': 51,
 'expected_forecast_rows': 51,
 'missing_ets_forecasts': np.int64(0),
 'missing_actual_sales': np.int64(0),
 'all_forecasts_finite': np.True_,
 'all_training_origins_before_validation': np.True_,
 'all_forecasts_nonnegative': np.True_}

In [71]:
# Calculate ETS forecast errors

ets_rolling_forecasts_df["ets_error"] = (
    ets_rolling_forecasts_df["ets_forecast"]
    - ets_rolling_forecasts_df["total_sales"]
)

ets_rolling_forecasts_df["ets_absolute_error"] = (
    ets_rolling_forecasts_df["ets_error"].abs()
)

ets_rolling_forecasts_df[
    [
        "window_id",
        "train_end",
        "month",
        "total_sales",
        "ets_forecast",
        "ets_error",
        "ets_absolute_error"
    ]
].head(10)

,window_id,train_end,month,total_sales,ets_forecast,ets_error,ets_absolute_error
0,1,2015-12-01,2016-01-01,2.397780e+07,2.346665e+07,-5.111597e+05,5.111597e+05
1,1,2015-12-01,2016-02-01,2.194741e+07,1.930117e+07,-2.646236e+06,2.646236e+06
2,1,2015-12-01,2016-03-01,2.313178e+07,2.421934e+07,1.087558e+06,1.087558e+06
3,2,2016-01-01,2016-02-01,2.194741e+07,1.936486e+07,-2.582550e+06,2.582550e+06
4,2,2016-01-01,2016-03-01,2.313178e+07,2.428877e+07,1.156988e+06,1.156988e+06
5,2,2016-01-01,2016-04-01,2.596303e+07,2.006642e+07,-5.896609e+06,5.896609e+06
6,3,2016-02-01,2016-03-01,2.313178e+07,2.485719e+07,1.725410e+06,1.725410e+06
7,3,2016-02-01,2016-04-01,2.596303e+07,2.062178e+07,-5.341241e+06,5.341241e+06
8,3,2016-02-01,2016-05-01,2.477943e+07,2.211446e+07,-2.664968e+06,2.664968e+06
9,4,2016-03-01,2016-04-01,2.596303e+07,2.026349e+07,-5.699538e+06,5.699538e+06


In [72]:
# Phase 3.13.3 — Compare ETS with existing models

actual = rolling_forecasts_df["total_sales"].reset_index(drop=True)

model_forecasts = {
    "Naive": rolling_forecasts_df["naive_forecast"].reset_index(drop=True),
    "Seasonal Naive": rolling_forecasts_df[
        "seasonal_naive_forecast"
    ].reset_index(drop=True),
    "50/50 Blended": blended_forecasts_df[
        "blended_forecast"
    ].reset_index(drop=True),
    "ETS Additive": ets_rolling_forecasts_df[
        "ets_forecast"
    ].reset_index(drop=True)
}

comparison_rows = []

for model_name, forecast in model_forecasts.items():
    errors = forecast - actual
    absolute_errors = errors.abs()

    comparison_rows.append({
        "model": model_name,
        "MAE": absolute_errors.mean(),
        "RMSE": np.sqrt((errors ** 2).mean()),
        "WAPE_pct": (
            absolute_errors.sum() / actual.abs().sum()
        ) * 100,
        "Bias_pct": (
            errors.sum() / actual.abs().sum()
        ) * 100
    })

ets_model_comparison_df = pd.DataFrame(comparison_rows)

ets_model_comparison_df.round(2)

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,Naive,2178014.34,2735640.76,8.80,-0.59
1,Seasonal Naive,3210528.38,4529398.49,12.98,-12.31
2,50/50 Blended,2067404.61,2737131.72,8.36,-6.45
3,ETS Additive,2098242.04,2542024.93,8.48,0.96


In [73]:
ets_comparison_qa = {
    "model_count": len(ets_model_comparison_df),
    "expected_models": 4,
    "unique_model_names": (
        ets_model_comparison_df["model"].nunique() == 4
    ),
    "missing_metrics": (
        ets_model_comparison_df[
            ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
        ].isna().sum().sum()
    ),
    "all_metrics_finite": np.isfinite(
        ets_model_comparison_df[
            ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
        ].to_numpy()
    ).all(),
    "same_actual_observation_count": (
        len(actual) == len(ets_rolling_forecasts_df) == 51
    )
}

ets_comparison_qa

{'model_count': 4,
 'expected_models': 4,
 'unique_model_names': True,
 'missing_metrics': np.int64(0),
 'all_metrics_finite': np.True_,
 'same_actual_observation_count': True}

In [74]:
# Phase 3.13.3 — Window-level ETS comparison

ets_window_comparison = (
    ets_rolling_forecasts_df
    .groupby("window_id")
    .apply(
        lambda g: pd.Series({
            "ets_wape_pct": (
                g["ets_absolute_error"].sum()
                / g["total_sales"].abs().sum()
            ) * 100
        }),
        include_groups=False
    )
    .reset_index()
)

# Add comparable baseline WAPE values
ets_window_comparison = ets_window_comparison.merge(
    window_comparison[
        [
            "window_id",
            "naive_wape_pct",
            "seasonal_naive_wape_pct",
            "blended_wape_pct"
        ]
    ],
    on="window_id",
    how="left",
    validate="one_to_one"
)

ets_window_comparison["ets_wins_vs_naive"] = (
    ets_window_comparison["ets_wape_pct"]
    < ets_window_comparison["naive_wape_pct"]
)

ets_window_comparison["ets_wins_vs_blend"] = (
    ets_window_comparison["ets_wape_pct"]
    < ets_window_comparison["blended_wape_pct"]
)

ets_window_comparison.round(2)# Phase 3.13.3 — Window-level ETS comparison

ets_window_comparison = (
    ets_rolling_forecasts_df
    .groupby("window_id")
    .apply(
        lambda g: pd.Series({
            "ets_wape_pct": (
                g["ets_absolute_error"].sum()
                / g["total_sales"].abs().sum()
            ) * 100
        }),
        include_groups=False
    )
    .reset_index()
)

# Add comparable baseline WAPE values
ets_window_comparison = ets_window_comparison.merge(
    window_comparison[
        [
            "window_id",
            "naive_wape_pct",
            "seasonal_naive_wape_pct",
            "blended_wape_pct"
        ]
    ],
    on="window_id",
    how="left",
    validate="one_to_one"
)

ets_window_comparison["ets_wins_vs_naive"] = (
    ets_window_comparison["ets_wape_pct"]
    < ets_window_comparison["naive_wape_pct"]
)

ets_window_comparison["ets_wins_vs_blend"] = (
    ets_window_comparison["ets_wape_pct"]
    < ets_window_comparison["blended_wape_pct"]
)

ets_window_comparison.round(2)

,window_id,ets_wape_pct,naive_wape_pct,seasonal_naive_wape_pct,blended_wape_pct,ets_wins_vs_naive,ets_wins_vs_blend
0,1,6.15,18.35,35.94,8.79,True,True
1,2,13.56,6.84,37.65,18.20,False,True
2,3,13.17,10.87,34.64,22.76,False,True
3,4,13.76,7.40,25.57,15.67,False,True
4,5,10.68,10.56,12.63,7.29,False,False
5,6,13.86,9.12,3.46,3.58,False,False
6,7,14.19,2.49,3.79,2.48,False,False
7,8,12.76,3.81,2.16,2.98,False,False
8,9,14.41,5.35,3.96,4.65,False,False
9,10,7.03,14.12,5.60,9.86,True,True


In [75]:
# Phase 3.13.3 — ETS window-level summary and QA

ets_window_summary = {
    "window_count": len(ets_window_comparison),

    "missing_values": int(
        ets_window_comparison.isna().sum().sum()
    ),

    "ets_wins_vs_naive": int(
        ets_window_comparison["ets_wins_vs_naive"].sum()
    ),

    "ets_wins_vs_blend": int(
        ets_window_comparison["ets_wins_vs_blend"].sum()
    ),

    "ets_mean_window_wape_pct": round(
        ets_window_comparison["ets_wape_pct"].mean(), 2
    ),

    "naive_mean_window_wape_pct": round(
        ets_window_comparison["naive_wape_pct"].mean(), 2
    ),

    "blended_mean_window_wape_pct": round(
        ets_window_comparison["blended_wape_pct"].mean(), 2
    ),

    "all_17_windows_present": (
        len(ets_window_comparison) == 17
        and ets_window_comparison["window_id"].nunique() == 17
    ),

    "all_ets_wape_finite": np.isfinite(
        ets_window_comparison["ets_wape_pct"].to_numpy()
    ).all()
}

ets_window_summary

{'window_count': 17,
 'missing_values': 0,
 'ets_wins_vs_naive': 7,
 'ets_wins_vs_blend': 12,
 'ets_mean_window_wape_pct': np.float64(8.69),
 'naive_mean_window_wape_pct': np.float64(8.78),
 'blended_mean_window_wape_pct': np.float64(8.33),
 'all_17_windows_present': True,
 'all_ets_wape_finite': np.True_}

### Phase 3.13.3 — Additive ETS Forecast Evaluation

**Objective:** Evaluate Holt-Winters Exponential Smoothing (ETS) against the Naive, Seasonal Naive, and 50/50 blended baselines using the established rolling-origin validation protocol.

**Model configuration**
- Additive trend
- Additive seasonality
- Seasonal period: 12 months
- Estimated initialization
- Optimized model parameters
- Forecast horizon: 3 months
- Minimum training history: 36 monthly observations

**Validation design**
- 17 chronological rolling-origin windows
- 51 forecast instances across the evaluation
- Each model uses only observations available before its forecast origin
- Forecast errors calculated as forecast minus actual Sales
- Evaluation metrics: MAE, RMSE, WAPE, and Bias

**Pooled rolling-origin results**

| Model | MAE | RMSE | WAPE | Bias |
|---|---:|---:|---:|---:|
| Naive | 2,178,014.34 | 2,735,640.76 | 8.80% | -0.59% |
| Seasonal Naive | 3,210,528.38 | 4,529,398.49 | 12.98% | -12.31% |
| 50/50 Blended | 2,067,404.61 | 2,737,131.72 | 8.36% | -6.45% |
| Additive ETS | 2,098,242.04 | 2,542,024.93 | 8.48% | +0.96% |

**Window-level findings**
- ETS achieved lower WAPE than Naive in 7 of 17 windows.
- ETS achieved lower WAPE than the 50/50 blend in 12 of 17 windows.
- Mean window WAPE was 8.69% for ETS, 8.78% for Naive, and 8.33% for the blend.
- All 17 windows were present, with zero missing values in the window comparison and finite ETS WAPE values.

**Interpretation**

The 50/50 blend delivered the lowest pooled MAE and WAPE. ETS delivered the lowest pooled RMSE and Bias closest to zero. Its window-level performance varied, and it did not consistently outperform the Naive baseline.

The results support retaining both the blend and ETS as candidate forecasting methods while preserving Naive as a simple benchmark. Model selection should consider multiple error metrics, performance across forecast origins, business costs, and the operational consequences of overforecasting and underforecasting.

**Limitations**
- Rolling windows overlap, so the 51 forecast instances are not independent observations.
- The May–July 2017 period also appears in the final rolling window and is not an untouched final test set.
- The results establish comparative performance only for the tested data, horizon, and model configurations; they do not guarantee future performance.
- Error patterns are descriptive and do not establish causal effects from promotions, holidays, or other retail drivers.

**Status:** Additive ETS rolling-origin evaluation completed; documentation and Git synchronization pending.


In [76]:
# Phase 3.13.4 — Damped-Trend ETS Diagnostic Fit

from statsmodels.tsa.holtwinters import ExponentialSmoothing
import numpy as np
import pandas as pd

# Use the first 36 months for training
damped_train = forecast_series.iloc[:36]

# The next three months are the diagnostic forecast period
damped_expected_months = forecast_series.index[36:39]

damped_ets_model = ExponentialSmoothing(
    damped_train,
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
)

damped_ets_fit = damped_ets_model.fit(optimized=True)

damped_diagnostic_forecast = damped_ets_fit.forecast(3)

damped_diagnostic_forecast.index = damped_expected_months

damped_diagnostic_qa = {
    "training_observations": len(damped_train),
    "forecast_observations": len(damped_diagnostic_forecast),
    "expected_forecast_observations": 3,
    "forecast_months_match": damped_diagnostic_forecast.index.equals(
        damped_expected_months
    ),
    "all_forecasts_finite": np.isfinite(
        damped_diagnostic_forecast.to_numpy()
    ).all(),
    "all_forecasts_nonnegative": (
        damped_diagnostic_forecast >= 0
    ).all()
}

print(damped_diagnostic_forecast)
print(damped_diagnostic_qa)

month
2016-01-01    2.262673e+07
2016-02-01    1.910118e+07
2016-03-01    2.330344e+07
dtype: float64
{'training_observations': 36, 'forecast_observations': 3, 'expected_forecast_observations': 3, 'forecast_months_match': True, 'all_forecasts_finite': np.True_, 'all_forecasts_nonnegative': np.True_}


d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [77]:
# Phase 3.13.4 — Rolling-Origin Damped ETS Forecasts

damped_rolling_forecasts = []

for window_id, window in rolling_windows_df.iterrows():

    train_end = window["train_end"]
    validation_start = window["validation_start"]
    validation_end = window["validation_end"]

    # Use only historical observations available at this origin
    damped_train = forecast_series.loc[
        forecast_series.index <= train_end
    ]

    # Actual values for the three-month validation window
    damped_actual = forecast_series.loc[
        (forecast_series.index >= validation_start)
        & (forecast_series.index <= validation_end)
    ]

    # Fit a fresh damped ETS model for this origin
    damped_model = ExponentialSmoothing(
        damped_train,
        trend="add",
        damped_trend=True,
        seasonal="add",
        seasonal_periods=12,
        initialization_method="estimated"
    )

    damped_fit = damped_model.fit(optimized=True)

    damped_forecast = damped_fit.forecast(
        len(damped_actual)
    )

    # Record each forecast and its actual value
    for month, forecast_value, actual_value in zip(
        damped_actual.index,
        damped_forecast.to_numpy(),
        damped_actual.to_numpy()
    ):
        damped_rolling_forecasts.append({
            "window_id": window_id + 1,
            "train_end": train_end,
            "month": month,
            "total_sales": actual_value,
            "damped_ets_forecast": forecast_value
        })

damped_rolling_forecasts_df = pd.DataFrame(
    damped_rolling_forecasts
)

damped_rolling_forecasts_df["damped_ets_error"] = (
    damped_rolling_forecasts_df["damped_ets_forecast"]
    - damped_rolling_forecasts_df["total_sales"]
)

damped_rolling_forecasts_df["damped_ets_absolute_error"] = (
    damped_rolling_forecasts_df["damped_ets_error"].abs()
)

damped_rolling_forecasts_df.head(10)

d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\GitHub\retail-sales-forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: 

,window_id,train_end,month,total_sales,damped_ets_forecast,damped_ets_error,damped_ets_absolute_error
0,1,2015-12-01,2016-01-01,2.397780e+07,2.262673e+07,-1.351078e+06,1.351078e+06
1,1,2015-12-01,2016-02-01,2.194741e+07,1.910118e+07,-2.846231e+06,2.846231e+06
2,1,2015-12-01,2016-03-01,2.313178e+07,2.330344e+07,1.716587e+05,1.716587e+05
3,2,2016-01-01,2016-02-01,2.194741e+07,1.959551e+07,-2.351900e+06,2.351900e+06
4,2,2016-01-01,2016-03-01,2.313178e+07,2.446527e+07,1.333487e+06,1.333487e+06
5,2,2016-01-01,2016-04-01,2.596303e+07,2.022635e+07,-5.736678e+06,5.736678e+06
6,3,2016-02-01,2016-03-01,2.313178e+07,2.507872e+07,1.946940e+06,1.946940e+06
7,3,2016-02-01,2016-04-01,2.596303e+07,2.086633e+07,-5.096695e+06,5.096695e+06
8,3,2016-02-01,2016-05-01,2.477943e+07,2.230039e+07,-2.479045e+06,2.479045e+06
9,4,2016-03-01,2016-04-01,2.596303e+07,2.042966e+07,-5.533361e+06,5.533361e+06


In [78]:
# Phase 3.13.4 — Damped ETS Forecast QA

damped_ets_qa = {
    "window_count": (
        damped_rolling_forecasts_df["window_id"].nunique()
    ),

    "expected_windows": 17,

    "forecast_rows": len(damped_rolling_forecasts_df),

    "expected_forecast_rows": 51,

    "missing_forecasts": int(
        damped_rolling_forecasts_df[
            "damped_ets_forecast"
        ].isna().sum()
    ),

    "missing_actual_sales": int(
        damped_rolling_forecasts_df[
            "total_sales"
        ].isna().sum()
    ),

    "all_forecasts_finite": np.isfinite(
        damped_rolling_forecasts_df[
            "damped_ets_forecast"
        ].to_numpy()
    ).all(),

    "all_forecasts_nonnegative": (
        damped_rolling_forecasts_df[
            "damped_ets_forecast"
        ] >= 0
    ).all(),

    "all_errors_reconcile": np.allclose(
        damped_rolling_forecasts_df["damped_ets_error"],
        damped_rolling_forecasts_df["damped_ets_forecast"]
        - damped_rolling_forecasts_df["total_sales"]
    ),

    "all_windows_have_three_forecasts": (
        damped_rolling_forecasts_df
        .groupby("window_id")
        .size()
        .eq(3)
        .all()
    )
}

print(damped_ets_qa)

{'window_count': 17, 'expected_windows': 17, 'forecast_rows': 51, 'expected_forecast_rows': 51, 'missing_forecasts': 0, 'missing_actual_sales': 0, 'all_forecasts_finite': np.True_, 'all_forecasts_nonnegative': np.True_, 'all_errors_reconcile': True, 'all_windows_have_three_forecasts': np.True_}


In [79]:
# Phase 3.13.4 — Damped ETS Forecast QA

damped_ets_qa = {
    "window_count": (
        damped_rolling_forecasts_df["window_id"].nunique()
    ),

    "expected_windows": 17,

    "forecast_rows": len(damped_rolling_forecasts_df),

    "expected_forecast_rows": 51,

    "missing_forecasts": int(
        damped_rolling_forecasts_df[
            "damped_ets_forecast"
        ].isna().sum()
    ),

    "missing_actual_sales": int(
        damped_rolling_forecasts_df[
            "total_sales"
        ].isna().sum()
    ),

    "all_forecasts_finite": np.isfinite(
        damped_rolling_forecasts_df[
            "damped_ets_forecast"
        ].to_numpy()
    ).all(),

    "all_forecasts_nonnegative": (
        damped_rolling_forecasts_df[
            "damped_ets_forecast"
        ] >= 0
    ).all(),

    "all_errors_reconcile": np.allclose(
        damped_rolling_forecasts_df["damped_ets_error"],
        damped_rolling_forecasts_df["damped_ets_forecast"]
        - damped_rolling_forecasts_df["total_sales"]
    ),

    "all_windows_have_three_forecasts": (
        damped_rolling_forecasts_df
        .groupby("window_id")
        .size()
        .eq(3)
        .all()
    )
}

print(damped_ets_qa)

{'window_count': 17, 'expected_windows': 17, 'forecast_rows': 51, 'expected_forecast_rows': 51, 'missing_forecasts': 0, 'missing_actual_sales': 0, 'all_forecasts_finite': np.True_, 'all_forecasts_nonnegative': np.True_, 'all_errors_reconcile': True, 'all_windows_have_three_forecasts': np.True_}


In [80]:
# Phase 3.13.4 — Damped ETS Rolling-Origin Metrics

damped_actual = damped_rolling_forecasts_df["total_sales"]
damped_predicted = damped_rolling_forecasts_df["damped_ets_forecast"]
damped_errors = damped_predicted - damped_actual

damped_mae = damped_errors.abs().mean()

damped_rmse = np.sqrt(
    (damped_errors ** 2).mean()
)

damped_wape_pct = (
    damped_errors.abs().sum()
    / damped_actual.abs().sum()
) * 100

damped_bias_pct = (
    damped_errors.sum()
    / damped_actual.abs().sum()
) * 100

damped_metrics = pd.DataFrame([{
    "model": "ETS Damped Trend",
    "MAE": damped_mae,
    "RMSE": damped_rmse,
    "WAPE_pct": damped_wape_pct,
    "Bias_pct": damped_bias_pct
}])

damped_metrics.round(2)

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,ETS Damped Trend,2118859.66,2549228.14,8.56,0.3


In [81]:
# Combine existing models with damped ETS

all_model_comparison_df = pd.concat(
    [
        ets_model_comparison_df,
        damped_metrics
    ],
    ignore_index=True
)

all_model_comparison_df.round(2)

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,Naive,2178014.34,2735640.76,8.80,-0.59
1,Seasonal Naive,3210528.38,4529398.49,12.98,-12.31
2,50/50 Blended,2067404.61,2737131.72,8.36,-6.45
3,ETS Additive,2098242.04,2542024.93,8.48,0.96
4,ETS Damped Trend,2118859.66,2549228.14,8.56,0.30


In [82]:
# Phase 3.13.4 — Window-level damped ETS comparison

damped_window_comparison = (
    damped_rolling_forecasts_df
    .groupby("window_id")
    .apply(
        lambda g: pd.Series({
            "damped_wape_pct": (
                g["damped_ets_absolute_error"].sum()
                / g["total_sales"].abs().sum()
            ) * 100
        }),
        include_groups=False
    )
    .reset_index()
)

damped_window_comparison = damped_window_comparison.merge(
    ets_window_comparison[
        ["window_id", "ets_wape_pct"]
    ],
    on="window_id",
    how="left",
    validate="one_to_one"
)

damped_window_comparison = damped_window_comparison.merge(
    window_comparison[
        ["window_id", "blended_wape_pct", "naive_wape_pct"]
    ],
    on="window_id",
    how="left",
    validate="one_to_one"
)

damped_window_comparison["damped_wins_vs_ets"] = (
    damped_window_comparison["damped_wape_pct"]
    < damped_window_comparison["ets_wape_pct"]
)

damped_window_comparison["damped_wins_vs_blend"] = (
    damped_window_comparison["damped_wape_pct"]
    < damped_window_comparison["blended_wape_pct"]
)

damped_window_comparison.round(2)

,window_id,damped_wape_pct,ets_wape_pct,blended_wape_pct,naive_wape_pct,damped_wins_vs_ets,damped_wins_vs_blend
0,1,6.33,6.15,8.79,18.35,False,True
1,2,13.26,13.56,18.20,6.84,True,True
2,3,12.89,13.17,22.76,10.87,True,True
3,4,13.51,13.76,15.67,7.40,True,True
4,5,11.68,10.68,7.29,10.56,False,False
5,6,15.56,13.86,3.58,9.12,False,False
6,7,14.68,14.19,2.48,2.49,False,False
7,8,12.27,12.76,2.98,3.81,True,False
8,9,12.98,14.41,4.65,5.35,True,False
9,10,3.25,7.03,9.86,14.12,True,True


In [83]:
damped_window_summary = {
    "window_count": len(damped_window_comparison),
    "missing_values": int(
        damped_window_comparison.isna().sum().sum()
    ),
    "damped_wins_vs_additive_ets": int(
        damped_window_comparison["damped_wins_vs_ets"].sum()
    ),
    "damped_wins_vs_blend": int(
        damped_window_comparison["damped_wins_vs_blend"].sum()
    ),
    "damped_mean_window_wape_pct": round(
        damped_window_comparison["damped_wape_pct"].mean(), 2
    ),
    "all_17_windows_present": (
        len(damped_window_comparison) == 17
        and damped_window_comparison["window_id"].nunique() == 17
    )
}

damped_window_summary

{'window_count': 17,
 'missing_values': 0,
 'damped_wins_vs_additive_ets': 8,
 'damped_wins_vs_blend': 9,
 'damped_mean_window_wape_pct': np.float64(8.78),
 'all_17_windows_present': True}

### Phase 3.13.4 — Damped-Trend ETS Evaluation

**Objective:** Evaluate whether damping the additive ETS trend improves three-month retail Sales forecasts compared with the existing additive ETS model and baseline approaches.

**Model configuration**
- Additive trend with `damped_trend=True`
- Additive seasonality
- Seasonal period: 12 months
- Estimated initialization and optimized parameters
- Forecast horizon: 3 months
- Minimum training history: 36 monthly observations

**Validation design**
- 17 chronological rolling-origin windows
- 51 forecast instances
- Forecasts generated using only data available before each validation window
- Error convention: forecast minus actual Sales
- Metrics: MAE, RMSE, WAPE, and Bias

**Pooled rolling-origin results**

| Model | MAE | RMSE | WAPE | Bias |
|---|---:|---:|---:|---:|
| Naive | 2,178,014.34 | 2,735,640.76 | 8.80% | -0.59% |
| Seasonal Naive | 3,210,528.38 | 4,529,398.49 | 12.98% | -12.31% |
| 50/50 Blended | 2,067,404.61 | 2,737,131.72 | 8.36% | -6.45% |
| ETS Additive | 2,098,242.04 | 2,542,024.93 | 8.48% | +0.96% |
| ETS Damped Trend | 2,118,859.66 | 2,549,228.14 | 8.56% | +0.30% |

**Window-level results**
- Damped ETS achieved lower WAPE than additive ETS in 8 of 17 windows.
- Damped ETS achieved lower WAPE than the 50/50 blend in 9 of 17 windows.
- Mean window WAPE: damped ETS 8.78%, additive ETS 8.69%, and blended baseline 8.33%.
- QA confirmed 17 windows, 51 forecasts, no missing forecasts or actuals, finite nonnegative forecasts, reconciled errors, and three forecasts per window.
- Window-level comparison QA confirmed zero missing values and all 17 windows present.

**Interpretation**

Damping the trend moved aggregate Bias closer to zero, from +0.96% for additive ETS to +0.30%. However, MAE and WAPE increased slightly, and RMSE also increased slightly. Damped ETS did not consistently outperform additive ETS across rolling windows.

The 50/50 blend retained the lowest pooled MAE and WAPE, while additive ETS retained the lowest pooled RMSE. Damped ETS is retained as an evaluated alternative rather than the leading model.

**Limitations**
- Rolling windows overlap; the 51 forecast instances are not independent observations.
- The May–July 2017 validation period overlaps with the final rolling window and is not an untouched test set.
- Results apply to the evaluated series, three-month horizon, and tested model configurations; they do not guarantee future performance.

**Status:** Damped-trend ETS evaluation completed. Documentation and Git synchronization pending.